# Structural Break (Real-Time) — pivotal feature bank + LightGBM

Design principles baked into this notebook:

1. **TS-AUC compares series cross-sectionally at a fixed `t`, but inference is per-series.**
   So every evidence feature must be *pivotal*: same null distribution regardless of the
   underlying DGP. We get this by transforming every online point through the **historical
   empirical CDF** (PIT), then through `Phi^-1`. Under "no break", `z_t ~ N(0,1)` for every
   series, whatever its marginal is.

2. **The AUC-optimal score is any monotone transform of `P(break already happened | data)`.**
   So we train a plain binary classifier (logloss) and use its probability directly.
   A pure function of `t` is a constant within a t-slice => `t` is a free feature that can
   only help through interactions.

3. **Feature bank = (what changed) x (how to aggregate under unknown tau).**
   - what changed: after the Phase 1 cull, **scale and shape** -- every family that
     measured location, dependence or trend scored at the permutation-null floor and
     was archived (`tools/archived_families.py`).
   - how to aggregate: expanding, EWMA (3 decays in `ALPHAS`), trailing windows
     (3 lengths in `WINDOWS`), Page-CUSUM (3 drifts in `DRIFTS`). Multi-scale replaces
     the O(t) "max over candidate tau".

4. **Four channels, two registered banks.** `ChannelState` is maintained for all of
   `CH = ("z", "z2", "e", "e2")`:
   - `z`  = PIT-normal of the raw value,     `z2` = `(z^2 - 1)/sqrt(2)` -> its scale
   - `e`  = PIT-normal of the AR(p) residual, `e2` = the same, for the residual
   Only `z2` / `e2` are registered as ChannelBankFamily instances; `z` and `e` are read
   by `LMomentsFamily` and the two-sample families, so their channels stay live even
   though their banks are gone.

5. **Context features were cut.** The `h_*` block (historical kurtosis, AR coefs, sd/MAD,
   n_H) was meant to let LightGBM learn *how much to trust* each evidence feature. It never
   paid for itself in ablation, so it is archived along with the rest of the cull.
   `HistoricalRef.context` is still computed (it is a handful of floats, once per series)
   in case the block is restored.

Everything is O(1) per online step and the *same* extractor object is used by `train()` and
`infer()`, so there is no train/serve skew.


## Setup

Paste your own token below.


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [36]:
#pip install crunch-cli --upgrade --quiet --progress-bar off
#crunch setup structural-break-real-time junior-mule --token <YOUR_CRUNCH_TOKEN>

In [4]:
# @crunch/keep:on
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"
os.environ["OMP_WAIT_POLICY"] = "PASSIVE"
# @crunch/keep:off
import bisect
import shutil
import copy
import math
import os
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import special as _sp

def ndtri(x):                 # plain-Python wrapper: scipy's ufunc has __module__ = None,
    return _sp.ndtri(x)       # so cloudpickle would mis-resolve it to __mp_main__ in a
                              # loky worker. The wrapper pickles by value and carries a
                              # proper module reference to scipy.special instead.
from sklearn.metrics import roc_auc_score
from scipy.special import stdtr as _stdtr, ndtri as _ndtri, gammaln as _gln
from catboost import CatBoostClassifier


In [5]:
import crunch

crunch_tools = crunch.load_notebook()


loaded crunch tools for module: <module '__main__'>

cli version: 12.0.3
available ram: 16.00 gb
available cpu: 10 core
----


/Users/balendu/Desktop/structural/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
train_data, test_data = crunch_tools.load_data()
print("train series:", len(train_data))


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

## 1. The feature engine

`HistoricalRef` does the one-off O(n_H) work on the historical segment:
sorted array + `Phi^-1` lookup table for the PIT, robust moments, AR(p) coefficients by
Yule-Walker, the residual PIT table, and (new) the deterministic PIT-normal quantile grids
`z_ref`/`e_ref` used as the fixed "historical sample" side of every Tier 1-4 two-sample test.

**Feature registry, not a hand-maintained list.** A `FeatureFamily` is one self-contained
statistic (or small group of related statistics): it declares its own feature names, does any
one-off historical setup, and knows how to turn the shared per-step state into its own output
vector. A module-level `REGISTRY: list[FeatureFamily]` is the single source of truth --
`FEATURES` and `N_FEATURES` are *derived* from it, never hand-written, and `StreamFeatures`
is a thin driver that loops the registry and asserts each family returns exactly as many
values as `family.names()` promises. Adding a feature is: write one class, append it to
`REGISTRY`. Nothing else changes.

Per-channel trackers (ring buffer, window sums, EWMA, CUSUM state) live in a shared
`ChannelState`, pushed once per step by the driver; any family needing a channel reads it off
a per-step `Ctx` instead of maintaining its own copy. `Ctx.sorted_window(channel, W)` memoizes
the sorted trailing window for the rest of the step, which matters because most of the
surviving registry is two-sample tests that all want the same sorted view.

To add a feature: write a `FeatureFamily` subclass and append an instance to `REGISTRY`
(section 1d/1f-1i below). No positional-agreement assertion to keep in sync by hand -- the
mismatch check is per-family and fails immediately if a family's `update()` doesn't return
exactly as many values as its `names()`.


In [7]:
# =====================================================================
#  HISTORICAL REFERENCE  (one pass over the historical segment)
# =====================================================================
def _acf(x: np.ndarray, nlags: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    n = x.size
    d = float(np.dot(x, x))
    if d <= 0.0:
        return np.zeros(nlags)
    return np.array([float(np.dot(x[k:], x[:n - k])) / d for k in range(1, nlags + 1)])


def _yule_walker(r: np.ndarray, p: int) -> np.ndarray:
    R = np.eye(p)
    for i in range(p):
        for j in range(p):
            if i != j:
                R[i, j] = r[abs(i - j) - 1]
    try:
        a = np.linalg.solve(R, r[:p])
    except np.linalg.LinAlgError:
        a = np.zeros(p)
    if not np.all(np.isfinite(a)):
        a = np.zeros(p)
    return a


def _pit_z_of(values: np.ndarray) -> np.ndarray:
    """Map an array to its own PIT-normal scores (used for historical diagnostics)."""
    n = values.size
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(n, dtype=np.float64)
    return ndtri((ranks + 0.5) / n)


def _l_moments(sorted_x: np.ndarray):
    """Hosking (1990) unbiased sample L-moments l1..l4 from a SORTED array."""
    n = sorted_x.size
    if n < 4:
        return (0.0, 1e-6, 0.0, 0.0)
    x = sorted_x
    i = np.arange(1, n + 1, dtype=np.float64)
    b0 = x.mean()
    b1 = np.sum((i - 1) / (n - 1) * x) / n
    b2 = np.sum((i - 1) * (i - 2) / ((n - 1) * (n - 2)) * x) / n if n > 2 else 0.0
    b3 = np.sum((i - 1) * (i - 2) * (i - 3) / ((n - 1) * (n - 2) * (n - 3)) * x) / n if n > 3 else 0.0
    l1 = b0
    l2 = 2 * b1 - b0
    l3 = 6 * b2 - 6 * b1 + b0
    l4 = 20 * b3 - 30 * b2 + 12 * b1 - b0
    return (float(l1), float(max(l2, 1e-12)), float(l3), float(l4))


class HistoricalRef:
    """Everything derived once from the (break-free) historical segment."""

    def __init__(self, x_hist):
        xh = np.asarray(x_hist, dtype=np.float64).ravel()
        xh = xh[np.isfinite(xh)]
        if xh.size < 4 * AR_P + 8:                       # pathological guard
            xh = np.concatenate([xh, np.zeros(4 * AR_P + 8 - xh.size)])

        n = xh.size
        self.n = n
        self.x_series = xh[np.isfinite(xh)]          # ordered history, used by the GARCH families
        self.mu = float(xh.mean())
        self.sd = max(float(xh.std(ddof=1)), 1e-12)
        med = float(np.median(xh))
        self.mad = max(1.4826 * float(np.median(np.abs(xh - med))), 1e-12)
        self.xmin = float(xh.min())
        self.xmax = float(xh.max())

        # ---- PIT table for the raw value -----------------------------
        self.xs_l = np.sort(xh).tolist()
        self.ztab = ndtri((np.arange(n + 1) + 0.5) / (n + 1)).tolist()

        # ---- standardized moments / tail rates -----------------------
        zr = (xh - self.mu) / self.sd
        self.skew = float((zr ** 3).mean())
        self.kurt = float((zr ** 4).mean() - 3.0)
        self.p2 = max(float((np.abs(zr) > 2.0).mean()), 1.0 / (n + 1))
        self.p3 = max(float((np.abs(zr) > 3.0).mean()), 1.0 / (n + 1))

        # ---- AR(p) whitening ----------------------------------------
        r = _acf(xh, max(AR_P, 2))
        self.acf1, self.acf2 = float(r[0]), float(r[1])
        a = _yule_walker(r, AR_P)
        self.ar_l = [float(v) for v in a]

        c = xh - self.mu
        res = c[AR_P:].copy()
        for k in range(AR_P):
            res -= self.ar_l[k] * c[AR_P - 1 - k: c.size - 1 - k]
        self.rsd = max(float(res.std(ddof=1)), 1e-12)

        m = res.size
        self.es_l = np.sort(res).tolist()
        self.etab = ndtri((np.arange(m + 1) + 0.5) / (m + 1)).tolist()

        # ---- lag seeds so the online stream starts warm ---------------
        zh = _pit_z_of(xh)
        rz = _acf(zh, 2)
        self.zacf1, self.zacf2 = float(rz[0]), float(rz[1])
        self.z_seed1 = float(zh[-1])
        self.z_seed2 = float(zh[-2])
        self.tail_x = [float(v) for v in xh[-AR_P:]]

        eh = _pit_z_of(res)
        self.e_seed1 = float(eh[-1])
        self.e_seed2 = float(eh[-2])

        # ---- context block (constant for the whole series) ------------
        self.context = [
            float(n), math.log(n),
            self.skew, self.kurt, self.acf1, self.acf2,
            self.ar_l[0], self.ar_l[1], self.ar_l[2],
            self.rsd / self.sd, self.sd / self.mad,
            self.p2, self.p3, self.zacf1,
        ]

        # ---- Tier 1-4: deterministic PIT-normal quantile grids, used as the
        # fixed "historical sample" side of every two-sample test (see 1b markdown).
        # REF_MAX lives in the CONFIG cell below: the per-family
        # cache key (section 2b) has to hash it.
        _zr = np.asarray(self.ztab[1:-1], dtype=np.float64)
        _er = np.asarray(self.etab[1:-1], dtype=np.float64)
        if _zr.size > REF_MAX:
            _zr = np.ascontiguousarray(_zr[np.linspace(0, _zr.size - 1, REF_MAX).astype(int)])
        if _er.size > REF_MAX:
            _er = np.ascontiguousarray(_er[np.linspace(0, _er.size - 1, REF_MAX).astype(int)])
        self.z_ref, self.e_ref = _zr, _er
        self.z_ref_lmom = _l_moments(np.sort(self.z_ref))
        self.e_ref_lmom = _l_moments(np.sort(self.e_ref))


In [10]:
# =====================================================================
#  CONFIG
# =====================================================================
# @crunch/keep:on
WINDOWS = (32, 64, 256)      # trailing windows = candidate post-break lengths
TWOSAMPLE_CHANNELS = ("e",)      # was ("z","e"); all 30 top diagnostic features were e-channel
ALPHAS  = (0.02, 0.08, 0.25)     # EWMA decays: slow / medium / fast
DRIFTS  = (0.25, 0.75, 1.50)     # Page-CUSUM reference drifts, in sigma units
ACF_W   = 64                     # window for streaming autocorrelation
SLOPE_W = 64                     # window for streaming regression slope
SIGN_W  = (32, 128)              # windows for the sign / runs statistics
TAIL_W  = 64                     # window for tail-exceedance counts
BIN_W   = 128                    # window for the binned chi-square (shape change)
NBINS   = 8
AR_P    = 3                      # AR order for the whitening channel
REF_MAX = 256                    # knots in the deterministic PIT-normal
# @crunch/keep:off
                                 # quantile grid that is the reference side
                                 # of every two-sample test. The grid is
                                 # smooth, not raw data, so 256 knots
                                 # reproduce it well inside the sampling
                                 # noise of any window of length <= 256.
                                 # Module-level so the per-family cache key
                                 # (section 2b) can hash it.

# @crunch/keep:on
CH = ("z", "z2", "e", "e2")      # channels: loc / scale / loc-of-residual / scale-of-residual
LBUF = max(max(WINDOWS), BIN_W, ACF_W, SLOPE_W, max(SIGN_W), TAIL_W) + 1
INV_SQRT2 = 0.7071067811865476
# @crunch/keep:off

# FEATURES / N_FEATURES are no longer hand-written here -- they're derived from
# REGISTRY once every FeatureFamily has been defined and registered (see the end
# of section 1i, after the Tier 1-4 families). Everything between here and there
# is registry plumbing + the actual family classes.


### 1c. Registry infrastructure

`ChannelState` is the shared incremental tracker for one channel (ring buffer, per-`WINDOWS`
running sums, exp sum, per-`ALPHAS` EWMA, per-`DRIFTS` CUSUM) -- pushed once per step by the
driver, read by any family that needs this channel (the old code recomputed / re-read this ad
hoc in several places; now it's pushed once, read many times).

`Ctx` is what the driver hands each family's `update()`: this step's `t`/`n`, the 4 channel
values, `zr` (read by `TailFamily`), `u`, the `channels` dict, and `HistoricalRef`.
`u` currently has **no consumer** -- `Chi2Family` was its only reader and was cut in the
Phase 1 cull. It is left on `Ctx` because it is one float per step and it is what any
new binned/EDF family would want.
`Ctx.sorted_window(channel, W)` lazily sorts a trailing window and **caches it for the rest of
this step** -- several Tier 1-4 families want the same (channel, W) sorted view in a single
step, so this avoids re-sorting it once per family.

`stride_for(cost, t)` is a pure function of `t` (never of series content), mirroring
`PREDICT_STRIDE`/`_stride_for` in `infer()` -- the same argument applies: every series is
equally stale at a given `t`, so within-`t` ranking (all TS-AUC cares about) is unaffected.
`"O1"` families run every step (unchanged from before); `"sorted"`/`"heavy"` families -- the
new Tier 1-4 families that need a trailing window slice or a sorted view -- recompute on a
coarser, dense-early/sparse-late schedule and hold their previous value between recomputes.
This is also what bounds the cost of `Ctx.sorted_window`: since it's only reached when a strided
family actually recomputes, the window only gets re-sorted at that same reduced cadence, not
every step -- a full incrementally-maintained sorted structure (`bisect.insort`/evict) would
shave further, and is a reasonable next step if the stride schedule above turns out to be too
coarse, but wasn't necessary to hit the O(W log W)-at-a-stride budget this refactor targets.


In [8]:
# @crunch/keep:on
CH_IDX = {name: i for i, name in enumerate(CH)}  # z:0 z2:1 e:2 e2:3
# @crunch/keep:off


class ChannelState:
    """Ring buffer + exp/EWMA/window/CUSUM incremental trackers for one channel.
    Owned by the driver, pushed once per step; any family needing this channel
    reads it off `ctx` instead of recomputing it.

    Two copies of the ring buffer are kept on purpose. `buf` is a Python list: the
    running-sum update reads one scalar per window per step, and a list index is
    cheaper than a numpy 0-d unbox. `abuf` is the numpy mirror that `window()`
    slices, which replaces a 256-element list comprehension per window per step."""

    def __init__(self):
        L = LBUF
        self.buf = [0.0] * L
        self.abuf = np.zeros(L, dtype=np.float64)
        self.rs = [0.0] * len(WINDOWS)
        self.exp_s = 0.0
        self.ew_m = [0.0] * len(ALPHAS)
        self.ew_w2 = [0.0] * len(ALPHAS)
        self.gp = [0.0] * len(DRIFTS)
        self.gn = [0.0] * len(DRIFTS)

    def push(self, v: float, t: int) -> None:
        L = LBUF
        i = t % L
        b = self.buf
        rs = self.rs
        for k in range(len(WINDOWS)):
            rs[k] += v - b[(i - WINDOWS[k]) % L]
        b[i] = v
        self.abuf[i] = v
        self.exp_s += v
        m, w2 = self.ew_m, self.ew_w2
        for k in range(len(ALPHAS)):
            a = ALPHAS[k]
            m[k] = (1.0 - a) * m[k] + a * v
            w2[k] = (1.0 - a) * (1.0 - a) * w2[k] + a * a
        gp, gn = self.gp, self.gn
        for k in range(len(DRIFTS)):
            d = DRIFTS[k]
            p = gp[k] + v - d
            gp[k] = p if p > 0.0 else 0.0
            q = gn[k] - v - d
            gn[k] = q if q > 0.0 else 0.0

    def window(self, W: int, t: int) -> np.ndarray:
        """Last min(W, t+1) raw pushed values, chronological order."""
        n = t + 1
        m = W if n > W else n
        L = LBUF
        i = t % L
        s = i - m + 1
        if s >= 0:
            return self.abuf[s:i + 1].copy()
        return np.concatenate([self.abuf[s % L:], self.abuf[:i + 1]])


class Ctx:
    __slots__ = ("t", "n", "vals", "zr", "u", "channels", "h",
                 "_sort_cache", "_var_cache", "_rank_cache")

    def __init__(self, t, n, vals, zr, u, channels, h):
        self.t = t
        self.n = n
        self.vals = vals              # dict: "z"->float, "z2"->..., "e"->..., "e2"->...
        self.zr = zr
        self.u = u
        self.channels = channels      # dict: "z"->ChannelState, ...
        self.h = h
        self._sort_cache = {}
        self._var_cache = {}
        self._rank_cache = {}

    def sorted_window(self, channel: str, W: int) -> np.ndarray:
        key = (channel, W)
        c = self._sort_cache.get(key)
        if c is None:
            w = self.channels[channel].window(W, self.t)
            c = self._sort_cache[key] = np.sort(w)
        return c

    def window_var(self, channel: str, W: int) -> float:
        """Sample variance (ddof=1) of the trailing window, memoized for the step.
        ftest and bartlett both want it; it matches _mean_var()'s guards exactly."""
        key = (channel, W)
        v = self._var_cache.get(key)
        if v is None:
            w = self.sorted_window(channel, W)
            v = self._var_cache[key] = (max(float(w.var(ddof=1)), 1e-12)
                                        if w.size >= 2 else 1e-6)
        return v

    def window_ranks(self, channel: str, W: int, ref_sorted: np.ndarray) -> np.ndarray:
        """Tie-averaged ranks of the sorted trailing window inside (window ++ reference),
        memoized for the step. `ansari` and `mood` compute exactly this, off the same
        reference grid, so they share one computation.

        Keyed on (channel, W) only, which is safe because every caller is a
        RefCachedTwoSample and they all read the SAME reference (h.z_ref / h.e_ref via
        TwoSampleFamily.init). A family with its own reference -- `fligner` ranks
        |x - median| instead -- must not use this cache, and does not."""
        key = (channel, W)
        r = self._rank_cache.get(key)
        if r is None:
            w = self.sorted_window(channel, W)
            r = self._rank_cache[key] = _avg_ranks_of(w, w, ref_sorted)
        return r


def stride_for(cost: str, t: int) -> int:
    """Pure function of t only -- every series at step t is equally stale, so
    within-t ranking is unaffected. Same argument as PREDICT_STRIDE / _stride_for
    in infer(): free speed, no data leakage, no series-dependent staleness."""
    if cost == "O1":
        return 1
    if cost == "sorted":
        if t < 128:
            return 1
        if t < 512:
            return 4
        return 16
    # "heavy"
    if t < 128:
        return 2
    if t < 512:
        return 8
    return 32


def _safe(v: float) -> float:
    return v if math.isfinite(v) else 0.0


def _safe_ratio(num: float, den: float) -> float:
    if den <= 0.0 or not math.isfinite(num) or not math.isfinite(den):
        return 0.0
    v = num / den
    return v if math.isfinite(v) else 0.0


class FeatureFamily:
    """Base class for one entry in REGISTRY. init() does one-off O(n_H) work on
    the historical segment; update() consumes ctx (the shared per-step state) and
    returns len(self.names()) floats, in emit order.

    IMPORTANT: REGISTRY holds one prototype *instance* per family (so a family
    with constructor args, like ChannelBankFamily(channel), only needs to be
    built once). init()/update() mutate `self` (e.g. TailFamily's ring buffers and
    running counts), so every StreamFeatures MUST work on its own
    `clone()` of each family, never the REGISTRY prototype directly --
    build_dataset() runs StreamFeatures for different series concurrently
    (joblib threading backend), and sharing mutable family state across those
    threads would silently corrupt features across series."""
    name: str = "base"
    channels: tuple = ("z", "e")
    cost: str = "O1"
    version: int = 1                  # bump by hand after editing this family's body:
                                      # it is part of the per-family cache key (2b), and
                                      # inspect.getsource() usually cannot see a class
                                      # defined in a notebook cell, so `version` is the
                                      # only thing that invalidates stale columns.

    def names(self) -> List[str]:
        raise NotImplementedError

    def init(self, hist: HistoricalRef) -> None:
        pass

    def update(self, ctx: Ctx) -> List[float]:
        raise NotImplementedError

    def clone(self) -> "FeatureFamily":
        return copy.deepcopy(self)


# @crunch/keep:on
REGISTRY: List[FeatureFamily] = []
# @crunch/keep:off


NameError: name 'CH' is not defined

### 1d. Porting the existing bank into families (behavior-preserving)

Same arithmetic as the pre-registry code, relocated into `FeatureFamily` classes.

The Phase 1 cull removed `ContrastFamily`, `AutocorrFamily`, `SignRunsFamily`,
`Chi2Family`, `SlopeFamily` and `ContextFamily` from this cell, and unregistered the `z` /
`e` channel banks -- all of them sat at or below the permutation-null cutoff. Their source is
in `tools/archived_families.py` (offline parts bin, never imported), with restore instructions
at the top of that file.

| Family | Reads from `ctx` | Emits |
|---|---|---|
| `ChannelBankFamily(channel)` x2 (`z2`, `e2`) | `channels[c]` | exp / EWMA x3 / window x3 / CUSUM x3 (10 each) |
| `TailFamily` | `zr` | `tail{2,3}_w64`, `tail{2,3}_exp`, `beyond_range_exp`, `max_absz_exp` |
| `TimeFamily` | `t` | `t_step`, `t_log` -- **always kept**, exempt from every prune rule |

`ChannelState` is still built for all four channels in `CH`, not just the two registered
banks: `LMomentsFamily` reads `z` and `e`, and the two-sample families read
`TWOSAMPLE_CHANNELS`. Pruning a bank is not the same as pruning its channel.


In [12]:
class ChannelBankFamily(FeatureFamily):
    """exp / EWMA(3) / window(4) / CUSUM(3) bank for one channel."""

    def __init__(self, channel: str):
        self.channel = channel
        self.name = channel
        self.channels = (channel,)

    def names(self):
        p = self.channel
        n = [p + "_exp"]
        n += ["%s_ew%g" % (p, a) for a in ALPHAS]
        n += ["%s_w%d" % (p, W) for W in WINDOWS]
        n += ["%s_cusum%g" % (p, d) for d in DRIFTS]
        return n

    def update(self, ctx: Ctx):
        cs = ctx.channels[self.channel]
        n = ctx.n
        out = [cs.exp_s / math.sqrt(n)]
        for k in range(len(ALPHAS)):
            out.append(cs.ew_m[k] / math.sqrt(cs.ew_w2[k] + 1e-18))
        for k in range(len(WINDOWS)):
            nw = WINDOWS[k] if n > WINDOWS[k] else n
            out.append(cs.rs[k] / math.sqrt(nw))
        for k in range(len(DRIFTS)):
            out.append(cs.gp[k] if cs.gp[k] > cs.gn[k] else cs.gn[k])
        return out


class TailFamily(FeatureFamily):
    name = "tail"
    channels = ("z",)

    def init(self, hist: HistoricalRef):
        L = LBUF
        self.t2buf = [0.0] * L
        self.t3buf = [0.0] * L
        self.t2w = self.t3w = self.t2e = self.t3e = 0.0
        self.beyond = 0.0
        self.max_absz = 0.0
        self.p2, self.p3 = hist.p2, hist.p3
        self.xmin, self.xmax, self.hn = hist.xmin, hist.xmax, hist.n
        self.mu, self.sd = hist.mu, hist.sd

    def names(self):
        out = []
        for thr in (2, 3):
            out += ["tail%d_w%d" % (thr, TAIL_W), "tail%d_exp" % thr]
        out += ["beyond_range_exp", "max_absz_exp"]
        return out

    def update(self, ctx: Ctx):
        zr = ctx.zr
        t, n = ctx.t, ctx.n
        L = LBUF
        i = t % L
        az = zr if zr >= 0.0 else -zr
        e2 = 1.0 if az > 2.0 else 0.0
        e3 = 1.0 if az > 3.0 else 0.0
        self.t2w += e2 - self.t2buf[(i - TAIL_W) % L]
        self.t3w += e3 - self.t3buf[(i - TAIL_W) % L]
        self.t2buf[i] = e2
        self.t3buf[i] = e3
        self.t2e += e2
        self.t3e += e3
        nw = TAIL_W if n > TAIL_W else n
        out = []
        for cnt_w, cnt_e, p in ((self.t2w, self.t2e, self.p2), (self.t3w, self.t3e, self.p3)):
            out.append((cnt_w - nw * p) / math.sqrt(nw * p * (1.0 - p) + 1e-12))
            out.append((cnt_e - n * p) / math.sqrt(n * p * (1.0 - p) + 1e-12))
        xv = self.mu + zr * self.sd
        if xv < self.xmin or xv > self.xmax:
            self.beyond += 1.0
        pb_rate = 2.0 / (self.hn + 1.0)
        out.append((self.beyond - n * pb_rate) / math.sqrt(n * pb_rate * (1.0 - pb_rate) + 1e-12))
        if az > self.max_absz:
            self.max_absz = az
        out.append(self.max_absz)
        return out


class TimeFamily(FeatureFamily):
    """Pure functions of t. Structurally 0.5 TS-AUC alone -- kept always, exempt
    from every prune/ablate rule (only useful through interactions)."""
    name = "time"
    channels = ()

    def names(self):
        return ["t_step", "t_log"]

    def update(self, ctx: Ctx):
        return [float(ctx.t), math.log1p(ctx.t)]


# Only the two SQUARED (scale) channels are registered as feature banks. The raw
# location banks "z" / "e" were pruned -- but ChannelState is still maintained for
# every channel in CH, because LMomentsFamily reads "z" and "e" and the two-sample
# families read TWOSAMPLE_CHANNELS. Don't prune the channel plumbing along with the
# banks.
# @crunch/keep:on
BANK_CHANNELS = ("z2", "e2")

for _ch in BANK_CHANNELS:
    REGISTRY.append(ChannelBankFamily(_ch))
REGISTRY.append(TailFamily())
REGISTRY.append(TimeFamily())
# @crunch/keep:off

print("ported families registered:", len(REGISTRY),
      "| features so far:", sum(len(f.names()) for f in REGISTRY))


ported families registered: 4 | features so far: 28


In [13]:
# =====================================================================
#  STREAMING FEATURE EXTRACTOR  (thin registry driver)
# =====================================================================
class StreamFeatures:
    """Registry driver. `active_families=None` (default) runs every family in
    REGISTRY; pass a set of family `.name`s to restrict to a subset.

    Each family is `.clone()`-d from its REGISTRY prototype (see the note on
    FeatureFamily.clone()) so this instance's per-series state is fully its own --
    required for build_dataset()'s process/thread parallelism to be safe.

    Phase 2 changes, all of them exact:
      * only the ChannelStates some active family declares are built and pushed;
      * per-family output width is captured once in __init__ instead of calling
        f.names() and asserting on every step (the check moved to __init__);
      * every reference-side quantity is precomputed in each family's init().
    """

    def __init__(self, x_hist, active_families=None, registry=None):
        reg = REGISTRY if registry is None else registry
        h = self.h = HistoricalRef(x_hist)
        self.families = [f.clone() for f in reg
                          if active_families is None or f.name in active_families]
        for f in self.families:
            f.init(h)
        self.names = [nm for f in self.families for nm in f.names()]
        self._nout = [len(f.names()) for f in self.families]

        # Build only the channels some active family declares it reads. The union of
        # declared `channels` is the safe rule: ChannelBankFamily sets its own
        # .channels in __init__, TwoSampleFamily inherits TWOSAMPLE_CHANNELS, and
        # LMomentsFamily keeps the FeatureFamily default ("z", "e") -- which is what
        # keeps the "z" state alive after the Phase 1 bank cull.
        need = set()
        for f in self.families:
            need |= set(getattr(f, "channels", ()) or ())
        self.channels = {name: ChannelState() for name in CH if name in need}

        self.tx = list(h.tail_x)
        self.t = -1

    def update(self, x) -> List[float]:
        h = self.h
        self.t += 1
        t = self.t
        n = t + 1

        xv = float(x)
        if xv != xv or xv in (float("inf"), float("-inf")):
            xv = h.mu

        # ---- channel 0/1 : PIT of the raw value ----------------------
        j = bisect.bisect_left(h.xs_l, xv)
        z = h.ztab[j]
        u = (j + 0.5) / (h.n + 1.0)
        zr = (xv - h.mu) / h.sd

        # ---- channel 2/3 : PIT of the AR residual --------------------
        tx = self.tx
        rres = xv - h.mu
        for k in range(AR_P):
            rres -= h.ar_l[k] * (tx[AR_P - 1 - k] - h.mu)
        tx.pop(0)
        tx.append(xv)
        je = bisect.bisect_left(h.es_l, rres)
        ze = h.etab[je]

        vals = {"z": z, "z2": (z * z - 1.0) * INV_SQRT2,
                "e": ze, "e2": (ze * ze - 1.0) * INV_SQRT2}
        for name, cs in self.channels.items():
            cs.push(vals[name], t)

        ctx = Ctx(t, n, vals, zr, u, self.channels, h)

        out = []
        for f in self.families:
            stride = stride_for(f.cost, t)
            if stride == 1 or t % stride == 0 or not hasattr(f, "_last"):
                f._last = f.update(ctx)
            out.extend(f._last)

        if t == 0:
            # Every family computes at t == 0 whatever its stride, so this is the one
            # step where the width check is free. It replaces the original driver's
            # per-step `assert len(r) == len(f.names())`.
            for f, nout in zip(self.families, self._nout):
                if len(f._last) != nout:
                    raise AssertionError(
                        "%s.update() returned %d values, names() has %d" % (f.name, len(f._last), nout))
        return out


### 1f. Tier 1 -- the moment ladder

Every family below is a **two-sample test: historical reference (`h.e_ref`) vs. the trailing
window ending at `t`**, evaluated at every length in `WINDOWS` on the `TWOSAMPLE_CHANNELS`
channels (3 features per family). Where a closed-form null exists we emit the standardized
statistic directly -- that's what makes it pivotal.

What survived the Phase 1 cull here is exactly the **scale** block: F-test, Bartlett,
Levene (Brown-Forsythe) and Fligner-Killeen. The location tests (Welch t, Mann-Whitney) and
the normality/shape omnibus tests (D'Agostino skewness, Anscombe-Glynn kurtosis, Jarque-Bera,
D'Agostino-Pearson K²) all sat at the noise floor and are archived in
`tools/archived_families.py`.

`TwoSampleFamily` is the shared base: it derives `names()` from `(stat, channel, W)`, pulls
the window (raw or sorted, per subclass) via `Ctx`, and standardizes with a guard against
degenerate windows (returns 0.0, never NaN/inf).


In [14]:
def _mean_var(x: np.ndarray):
    n = x.size
    if n < 2:
        return 0.0, 1e-6
    mu = float(x.mean())
    v = float(x.var(ddof=1))
    return mu, max(v, 1e-12)


def _avg_ranks(x: np.ndarray) -> np.ndarray:
    """Tie-averaged (1-based) ranks of one array, via argsort over the whole array.

    No family in the pruned registry calls this any more. Every rank-based family
    (fligner, ansari, mood) now ranks with _avg_ranks_of(), which searchsorts into a
    reference that was sorted once in init() instead of concatenating and re-sorting
    window ++ reference on every evaluation; sigtuk needs stable-sort POSITIONS rather
    than averaged ranks and derives them the same way.

    It stays for two reasons: it is what a new family will reach for before there is a
    fast path for it, and it is the definition _avg_ranks_of() is checked against in
    tools/test_phase2.py."""
    n = x.size
    if n == 0:
        return np.empty(0, dtype=np.float64)
    order = np.argsort(x, kind="mergesort")
    s = x[order]
    neq = np.empty(n, dtype=bool)
    neq[0] = True
    if n > 1:
        np.not_equal(s[1:], s[:-1], out=neq[1:])
    starts = np.flatnonzero(neq)
    grp = np.cumsum(neq) - 1
    counts = np.diff(np.append(starts, n))
    avg = starts.astype(np.float64) + (counts.astype(np.float64) + 1.0) / 2.0
    out = np.empty(n, dtype=np.float64)
    out[order] = avg[grp]
    return out


def _guard(m: int, nr: int, min_m: int = 4) -> bool:
    return m >= min_m and nr >= min_m


class RefSub:
    """Fixed-size random subsample of a reference array, used by O(m*n_ref)
    pairwise-cost families (Tier 3 energy distance / MMD) so a large n_H
    doesn't blow up the per-step cost."""
    CAP = 192

    def __init__(self, ref: np.ndarray, seed: int = 0):
        if ref.size <= self.CAP:
            self.arr = ref
        else:
            rng = np.random.default_rng(seed)
            self.arr = rng.choice(ref, size=self.CAP, replace=False)


# =====================================================================
#  Exact fast paths shared by the two-sample families (Phase 2)
#
#  Every one of these is an algebraic rewrite, not an approximation: the only
#  difference from the code they replace is floating-point summation order.
#  `tools/reference_impl.py` holds the arithmetic they replaced and
#  tools/test_phase2.py diffs them at atol=rtol=1e-8.
# =====================================================================
# @crunch/keep:on
_COEF_CACHE = {}
# @crunch/keep:off


def _sorted_median(s: np.ndarray) -> float:
    """np.median(x) for an already-sorted 1-D array, without the internal partition."""
    m = s.size
    return 0.5 * (s[(m - 1) // 2] + s[m // 2])


def _sorted_percentile(s: np.ndarray, q: float) -> float:
    """np.percentile(s, q) (linear interpolation) for an already-sorted 1-D array."""
    pos = (s.size - 1) * q / 100.0
    lo = int(math.floor(pos))
    hi = min(lo + 1, s.size - 1)
    return s[lo] + (s[hi] - s[lo]) * (pos - lo)


def _avg_ranks_of(x_sorted_self: np.ndarray, x: np.ndarray,
                  other_sorted: np.ndarray) -> np.ndarray:
    """Average (tie-corrected, 1-based) ranks of `x` within the combined sample
    (x_sorted_self ++ other_sorted), by searchsorted instead of concatenate+argsort.

    rank = (#strictly-less + 1 + #less-or-equal) / 2, which is the midpoint of the
    tie block -- identical to _avg_ranks() on the concatenation."""
    less = np.searchsorted(x_sorted_self, x, "left") + np.searchsorted(other_sorted, x, "left")
    leq = np.searchsorted(x_sorted_self, x, "right") + np.searchsorted(other_sorted, x, "right")
    return 0.5 * (less + 1 + leq)


def _lmom_coefs(n: int):
    """Hosking b1/b2/b3 weight vectors for length n, built once per n."""
    c = _COEF_CACHE.get(("lmom", n))
    if c is None:
        i = np.arange(1, n + 1, dtype=np.float64)
        c1 = (i - 1) / (n - 1) / n
        c2 = (i - 1) * (i - 2) / ((n - 1) * (n - 2)) / n
        c3 = (i - 1) * (i - 2) * (i - 3) / ((n - 1) * (n - 2) * (n - 3)) / n
        c = _COEF_CACHE[("lmom", n)] = (c1, c2, c3)
    return c


def _l_moments_fast(x: np.ndarray):
    """_l_moments() with the three weight vectors precomputed (x must be SORTED)."""
    n = x.size
    if n < 4:
        return (0.0, 1e-6, 0.0, 0.0)
    c1, c2, c3 = _lmom_coefs(n)
    b0 = x.mean()
    b1 = float(np.dot(c1, x))
    b2 = float(np.dot(c2, x))
    b3 = float(np.dot(c3, x))
    l2 = 2 * b1 - b0
    l3 = 6 * b2 - 6 * b1 + b0
    l4 = 20 * b3 - 30 * b2 + 12 * b1 - b0
    return (float(b0), float(max(l2, 1e-12)), float(l3), float(l4))


def _vasicek_fast(sorted_x: np.ndarray) -> float:
    """Vasicek spacings entropy with the clipped index arrays built once per length."""
    m = sorted_x.size
    if m < 8:
        return 0.0
    key = ("vas", m)
    c = _COEF_CACHE.get(key)
    if c is None:
        q = max(1, int(round(math.sqrt(m))))
        idx = np.arange(m)
        c = _COEF_CACHE[key] = (np.clip(idx - q, 0, m - 1),
                                np.clip(idx + q, 0, m - 1), m / (2.0 * q))
    lo, hi, scale = c
    diffs = np.maximum(sorted_x[hi] - sorted_x[lo], 1e-9)
    return float(np.mean(np.log(scale * diffs)))


class TwoSampleFamily(FeatureFamily):
    cost = "sorted"
    needs_sorted = True
    channels = TWOSAMPLE_CHANNELS      # <-- add this line
    def names(self):
        return ["%s_%s_w%d" % (self.name, ch, W) for ch in self.channels for W in WINDOWS]

    def init(self, hist: HistoricalRef):
        self.ref = {"z": hist.z_ref, "e": hist.e_ref}

    def _window(self, ctx: Ctx, ch: str, W: int) -> np.ndarray:
        if self.needs_sorted:
            return ctx.sorted_window(ch, W)
        return ctx.channels[ch].window(W, ctx.t)

    def stat(self, w: np.ndarray, r: np.ndarray) -> float:
        raise NotImplementedError

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            ref = self.ref[ch]
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, ref)))
        return out


class RefCachedTwoSample(TwoSampleFamily):
    """TwoSampleFamily whose reference-side work is done once in init().

    Subclasses implement `stat_fast(ctx, ch, W)` and pull the window off `ctx`
    themselves, so they can use the memoized sorted window / variance / ranks
    instead of being handed a bare array."""

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_sorted = {}
        for ch in ("z", "e"):
            r = self.ref[ch]
            if np.any(np.diff(r) < 0):
                r = np.sort(r)
            self.ref_sorted[ch] = r

    def stat_fast(self, ctx: Ctx, ch: str, W: int) -> float:
        raise NotImplementedError

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                out.append(_safe(self.stat_fast(ctx, ch, W)))
        return out


class FTestFamily(RefCachedTwoSample):
    name = "ftest"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.v2 = {ch: _mean_var(self.ref[ch])[1] for ch in ("z", "e")}

    def stat_fast(self, ctx, ch, W):
        m, nr = ctx.sorted_window(ch, W).size, self.ref[ch].size
        if not _guard(m, nr):
            return 0.0
        v1 = ctx.window_var(ch, W)
        se = math.sqrt(2.0 / (m - 1) + 2.0 / (nr - 1))
        return math.log(v1 / self.v2[ch]) / se if se > 0 else 0.0


class BartlettFamily(FTestFamily):
    name = "bartlett"

    def stat_fast(self, ctx, ch, W):
        m, nr = ctx.sorted_window(ch, W).size, self.ref[ch].size
        if not _guard(m, nr):
            return 0.0
        v1, v2 = ctx.window_var(ch, W), self.v2[ch]
        sp2 = ((m - 1) * v1 + (nr - 1) * v2) / (m + nr - 2)
        b = (m + nr - 2) * math.log(sp2) - (m - 1) * math.log(v1) - (nr - 1) * math.log(v2)
        c = 1.0 + (1.0 / 3.0) * (1.0 / (m - 1) + 1.0 / (nr - 1) - 1.0 / (m + nr - 2))
        return ((b / c if c > 0 else 0.0) - 1.0) / math.sqrt(2.0)


class LeveneFamily(RefCachedTwoSample):
    """Brown-Forsythe variant (median-centered) -- robust to non-normality.
    The reference half of the statistic is constant, so it is precomputed."""
    name = "levene"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.rc = {}
        for ch in ("z", "e"):
            r = self.ref[ch]
            zr = np.abs(r - np.median(r))
            zb = zr.mean()
            self.rc[ch] = (zr.sum(), zb, float(((zr - zb) ** 2).sum()))

    def stat_fast(self, ctx, ch, W):
        w = ctx.sorted_window(ch, W)
        m, nr = w.size, self.ref[ch].size
        if not _guard(m, nr):
            return 0.0
        zr_sum, zbar_r, ss_r = self.rc[ch]
        zw = np.abs(w - _sorted_median(w))
        zbar_w = zw.mean()
        zbar = (zw.sum() + zr_sum) / (m + nr)
        ssb = m * (zbar_w - zbar) ** 2 + nr * (zbar_r - zbar) ** 2
        ssw = float(((zw - zbar_w) ** 2).sum() + ss_r)
        df2 = m + nr - 2
        if ssw <= 0 or df2 <= 0:
            return 0.0
        f = ssb / (ssw / df2)
        return (1.0 if zbar_w >= zbar_r else -1.0) * math.sqrt(max(f, 0.0))


class FlignerKilleenFamily(RefCachedTwoSample):
    """Rank-based (normal-scores) scale test -- distribution-free, robust to tails.

    Ranks |x - median| over both samples. That is a DIFFERENT ranking from the one
    ansari/mood share, so this family must not use Ctx.window_ranks."""
    name = "fligner"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ar = {}
        for ch in ("z", "e"):
            r = self.ref[ch]
            ar = np.sort(np.abs(r - np.median(r)))
            self.ar[ch] = (ar, np.searchsorted(ar, ar, "left"), np.searchsorted(ar, ar, "right"))

    def stat_fast(self, ctx, ch, W):
        w = ctx.sorted_window(ch, W)
        m = w.size
        ar, ar_l, ar_r = self.ar[ch]
        nr = ar.size
        if not _guard(m, nr):
            return 0.0
        aw = np.sort(np.abs(w - _sorted_median(w)))
        n = m + nr
        rk_w = _avg_ranks_of(aw, aw, ar)
        rk_r = 0.5 * (ar_l + np.searchsorted(aw, ar, "left") + 1
                      + ar_r + np.searchsorted(aw, ar, "right"))
        sw = ndtri(0.5 + rk_w / (2.0 * (n + 1)))
        sr = ndtri(0.5 + rk_r / (2.0 * (n + 1)))
        mean_score = (sw.sum() + sr.sum()) / n
        ss = ((sw - mean_score) ** 2).sum() + ((sr - mean_score) ** 2).sum()
        var = m * nr / n * (ss / (n - 1))
        return (sw.sum() - m * mean_score) / math.sqrt(max(var, 1e-9))


# @crunch/keep:on
REGISTRY.extend([
    FTestFamily(), BartlettFamily(), LeveneFamily(),
])
# @crunch/keep:off
print("Tier 1 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 1 registered | features so far: 37


### 1g. Tier 2 -- robust analogues

The diagnostic says we win on essentially **one** scale statistic (`e2_*`). This tier gives
the model several more, with different robustness/efficiency profiles: L-moments
(L-scale/L-skew/L-kurt -- linear combinations of order statistics, far lower sampling variance
than conventional moments under heavy tails), the quantile-based IQR ratio, and three
rank-based distribution-free scale tests (Ansari-Bradley, Mood, Siegel-Tukey -- the scale
analogue of Mann-Whitney).

Bowley skewness and Moors' octile kurtosis were cut in the Phase 1 cull (both shape, both at
the noise floor); they're in `tools/archived_families.py`.

`LMomentsFamily` is the one family here that is **not** a `TwoSampleFamily`: it inherits
`channels = ("z", "e")` from the base class, so it emits 2 channels x 3 statistics x 3 windows
= 18 features, and it is the reason the `z` `ChannelState` still has to be maintained.

Where no simple closed-form standard error exists (L-moments), the standardization is a
documented, dimensionally-consistent approximation (deviation from the reference value, scaled
by `~1/sqrt(m)`) rather than an exact asymptotic pivot -- a deliberate tradeoff, measured by
the permutation-null diagnostic in section 3 rather than assumed here.


In [15]:
class LMomentsFamily(FeatureFamily):
    """L-scale, L-skewness, L-kurtosis vs. the reference grid's L-moments.

    NOT a TwoSampleFamily: it inherits channels = ("z", "e") from FeatureFamily, so it
    emits 2 channels x 3 statistics x 3 windows = 18 features, and it is why the "z"
    ChannelState is still maintained after the Phase 1 bank cull."""
    name = "lmom"
    cost = "sorted"

    def init(self, hist: HistoricalRef):
        self.ref_lmom = {"z": hist.z_ref_lmom, "e": hist.e_ref_lmom}

    def names(self):
        out = []
        for ch in self.channels:
            for stat in ("lscale", "lskew", "lkurt"):
                for W in WINDOWS:
                    out.append("%s_%s_%s_w%d" % (self.name, stat, ch, W))
        return out

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            l1r, l2r, l3r, l4r = self.ref_lmom[ch]
            t3r, t4r = l3r / l2r, l4r / l2r
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                m = w.size
                if m < 8:
                    out += [0.0, 0.0, 0.0]
                    continue
                l1, l2, l3, l4 = _l_moments_fast(w)
                out.append(_safe(math.log(max(l2, 1e-9) / l2r) * math.sqrt(m - 1)))
                out.append(_safe((l3 / l2 - t3r) * math.sqrt(m)))
                out.append(_safe((l4 / l2 - t4r) * math.sqrt(m)))
        return out


class IQRRatioFamily(TwoSampleFamily):
    name = "iqr"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_iqr = {ch: float(np.subtract(*np.percentile(self.ref[ch], [75, 25])))
                         for ch in ("z", "e")}

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        # w arrives sorted (needs_sorted), so the quartiles are two interpolated reads
        iqr_w = float(_sorted_percentile(w, 75) - _sorted_percentile(w, 25))
        iqr_r = self.ref_iqr[self._cur_ch]
        if iqr_w <= 0 or iqr_r <= 0:
            return 0.0
        return math.log(iqr_w / iqr_r) * math.sqrt(max(m - 1, 1))

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


class _RankScaleFamily(RefCachedTwoSample):
    """Shared base for the rank-based scale tests that rank the window against the
    reference grid. `ansari` and `mood` differ only in the score applied to the ranks,
    so the ranks themselves are computed once per (channel, W) on the Ctx."""

    def w_ranks(self, ctx, ch, W):
        return ctx.sorted_window(ch, W), ctx.window_ranks(ch, W, self.ref_sorted[ch])


class AnsariBradleyFamily(_RankScaleFamily):
    """Rank-based scale test: score = min(rank, N+1-rank) (ends-inward ranking)."""
    name = "ansari"

    def stat_fast(self, ctx, ch, W):
        w, rk = self.w_ranks(ctx, ch, W)
        m, nr = w.size, self.ref[ch].size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        ab = np.minimum(rk, n + 1 - rk).sum()
        if n % 2 == 0:
            e = m * (n + 2) / 4.0
            var = m * nr * (n + 2) * (n - 2) / (48.0 * (n - 1))
        else:
            e = m * (n + 1) ** 2 / (4.0 * n)
            var = m * nr * (n + 1) * (3 + n * n) / (48.0 * n * n)
        return (ab - e) / math.sqrt(var) if var > 0 else 0.0


class MoodFamily(_RankScaleFamily):
    """Rank-based scale test: score = (rank - (N+1)/2)^2."""
    name = "mood"

    def stat_fast(self, ctx, ch, W):
        w, rk = self.w_ranks(ctx, ch, W)
        m, nr = w.size, self.ref[ch].size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        t = ((rk - (n + 1) / 2.0) ** 2).sum()
        e = m * (n * n - 1) / 12.0
        var = m * nr * (n + 1) * (n * n - 4) / 180.0
        return (t - e) / math.sqrt(var) if var > 0 else 0.0


class SiegelTukeyFamily(RefCachedTwoSample):
    """Ends-alternating ('zig-zag') ranking, then a Wilcoxon rank-sum on it.

    The zig-zag rank depends on the element's EXACT position under a stable sort of
    concat([window, reference]), not on its tie-averaged rank. For a sorted window,
    element i sits at position i + (number of reference values strictly below it):
    stability puts window values ahead of equal reference values, and ahead of each
    other in their existing order."""
    name = "sigtuk"

    def stat_fast(self, ctx, ch, W):
        w = ctx.sorted_window(ch, W)
        r = self.ref_sorted[ch]
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        n_lo = (n + 1) // 2
        pos = np.arange(m) + np.searchsorted(r, w, "left")
        st = np.where(pos < n_lo, 2.0 * pos + 1.0, 2.0 * (n - 1 - pos) + 2.0)
        mean = m * (n + 1) / 2.0
        var = m * nr * (n + 1) / 12.0
        return (st.sum() - mean) / math.sqrt(var) if var > 0 else 0.0


# @crunch/keep:on
REGISTRY.extend([
    LMomentsFamily(), SiegelTukeyFamily(),
])
# @crunch/keep:off
print("Tier 2 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 2 registered | features so far: 58


### 1h. Tier 3 -- omnibus / full-distribution distance

Wasserstein-1, energy distance, and MMD with an RBF kernel (bandwidth via the
median-pairwise-distance heuristic). Cramér-von Mises, Anderson-Darling, Hellinger and
Jensen-Shannon were cut in the Phase 1 cull -- see `tools/archived_families.py`. Note
`JSFamily` subclasses `HellingerFamily`, so restoring `js` means restoring `hell` too.

**KS is not reimplemented here.** The causal sup-KS scan near the bottom of the notebook
(`ks_scan_scores`) already is this test, fully causal, and is evaluated separately as a
standalone competing baseline against the whole LightGBM bank.

`cost = "heavy"`: these are `O(W log W)` to `O(W * n_ref_sub)` per evaluation (pairwise
distances for energy distance / MMD, capped via `RefSub` at 192 reference points so a large
`n_H` doesn't blow up the cost) -- they get the coarsest stride.

`MMDRbfFamily` inherits `init()` from `EnergyDistanceFamily`, which builds a `RefSub` for
**both** `z` and `e`; MMD's bandwidth is the median pairwise distance over the two
concatenated. That `z` reference is load-bearing even though `TWOSAMPLE_CHANNELS == ("e",)`.


In [16]:
class WassersteinFamily(RefCachedTwoSample):
    name = "wass"
    cost = "heavy"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self._qcache = {}

    def stat_fast(self, ctx, ch, W):
        w = ctx.sorted_window(ch, W)
        m, nr = w.size, self.ref[ch].size
        if not _guard(m, nr, 8):
            return 0.0
        # the reference quantiles depend only on (channel, window length), and there
        # are at most len(WINDOWS) distinct lengths per series
        key = (ch, m)
        rq = self._qcache.get(key)
        if rq is None:
            q = (np.arange(1, m + 1) - 0.5) / m
            rq = self._qcache[key] = np.quantile(self.ref[ch], q)
        return float(np.abs(w - rq).mean()) * math.sqrt(m)


class EnergyDistanceFamily(TwoSampleFamily):
    """Two-sample energy distance, O(m log m) instead of O(m * n_ref).

    For SORTED x, sum_ij |x_i - x_j| = 2 * sum_i (2i - m + 1) x_i, so the within-window
    term is one dot product. The cross term uses the reference prefix sums: for each
    w_i with k reference values <= it,
        sum_j |w_i - r_j| = w_i*k - cs[k] + (tot - cs[k]) - w_i*(nr - k).
    The reference-vs-reference term is constant and computed once in init()."""
    name = "energy"
    cost = "heavy"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.refsub = {ch: RefSub(hist.z_ref if ch == "z" else hist.e_ref).arr
                       for ch in ("z", "e")}
        self.pre = {}
        for ch in ("z", "e"):
            sub = self.refsub[ch]
            rs = np.sort(sub)
            rr = np.abs(sub[:, None] - sub[None, :]).mean()
            self.pre[ch] = (rs, np.concatenate([[0.0], np.cumsum(rs)]), rr)
        self._coef = {}

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            rs, cs, rr = self.pre[ch]
            nr, tot = rs.size, cs[-1]
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                m = w.size
                if m < 8:
                    out.append(0.0)
                    continue
                c = self._coef.get(m)
                if c is None:
                    c = self._coef[m] = (2.0 * np.arange(m) - m + 1.0) * 2.0 / (m * m)
                ww = float(np.dot(c, w))
                k = np.searchsorted(rs, w, "right")
                wr = float((w * k - cs[k] + (tot - cs[k]) - w * (nr - k)).sum()) / (m * nr)
                out.append(_safe(max(2 * wr - ww - rr, 0.0) * math.sqrt(m)))
        return out


class MMDRbfFamily(EnergyDistanceFamily):
    """MMD^2 with an RBF kernel. The reference-vs-reference kernel mean is constant,
    so only the window-window and window-reference blocks are built per step.

    The bandwidth is the median pairwise distance over BOTH channels' reference
    subsamples concatenated, so the "z" RefSub is load-bearing even when
    TWOSAMPLE_CHANNELS is ("e",) only."""
    name = "mmd"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        allref = np.concatenate([self.refsub["z"], self.refsub["e"]])
        d = np.abs(allref[:, None] - allref[None, :])
        med = np.median(d[d > 0]) if np.any(d > 0) else 1.0
        self.sigma2 = max(float(med) ** 2, 1e-6)
        s2 = self.sigma2
        self.krr = {ch: float(np.exp(-(self.refsub[ch][:, None] - self.refsub[ch][None, :]) ** 2
                                      / (2 * s2)).mean())
                    for ch in ("z", "e")}

    def update(self, ctx: Ctx):
        out = []
        s2 = self.sigma2
        for ch in self.channels:
            rs = self.refsub[ch]
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                m = w.size
                if m < 8:
                    out.append(0.0)
                    continue
                kww = np.exp(-(w[:, None] - w[None, :]) ** 2 / (2 * s2)).mean()
                kwr = np.exp(-(w[:, None] - rs[None, :]) ** 2 / (2 * s2)).mean()
                out.append(_safe(max(kww + self.krr[ch] - 2 * kwr, 0.0) * m))
        return out


# @crunch/keep:on
REGISTRY.extend([
    WassersteinFamily(), EnergyDistanceFamily(), MMDRbfFamily(),
])
# @crunch/keep:off
print("Tier 3 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 3 registered | features so far: 67


### 1i. Tier 4 -- properties no moment captures, then close out the registry

All that survived the Phase 1 cull in this tier is the Vasicek differential-entropy spacings
estimator. The Hill tail-index estimator, permutation entropy, order-1 ARCH-LM, Ljung-Box and
the R/S Hurst exponent are archived in `tools/archived_families.py`.

Once `entropy` is registered, `FEATURES` / `N_FEATURES` are derived from the complete
`REGISTRY` -- the only place they're computed, never hand-written. The pruned registry is
**17 families / 82 features**:

| | families |
|---|---|
| bank + structural | `e2`, `z2`, `tail`, `time` |
| scale tests | `ftest`, `bartlett`, `levene`, `fligner` |
| robust / rank scale | `lmom`, `iqr`, `ansari`, `mood`, `sigtuk` |
| distribution distance | `wass`, `energy`, `mmd` |
| entropy | `entropy` |

`ACTIVE_FAMILIES` is the speed/selection switch used by `infer()` and the crunch-submission
`train()` (sections 4/5): `None` means "every registered family", a `set` of family `.name`s
restricts to that subset. After the cull those are the same set, so it is `None` --
the registry itself is the selection. `resolve_active_families()` validates any subset against
`REGISTRY` up front so a typo'd family name fails loudly instead of silently dropping every
feature it would have added.

(`resolve_active_families` is defined **above** its first call in the cell below. It used to be
called two lines before its own `def`, which NameErrors on a fresh kernel.)


In [17]:
class DiffEntropyFamily(TwoSampleFamily):
    """Vasicek spacings estimator of differential entropy."""
    name = "entropy"
    cost = "heavy"
    needs_sorted = True

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_h = {ch: _vasicek_fast(np.sort(self.ref[ch])) for ch in ("z", "e")}

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        return (_vasicek_fast(w) - self.ref_h[self._cur_ch]) * math.sqrt(m)

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


# @crunch/keep:on
REGISTRY.extend([
    DiffEntropyFamily(),
])
# @crunch/keep:off

# =====================================================================
#  REGISTRY IS NOW COMPLETE -- derive FEATURES / N_FEATURES from it
# =====================================================================
# @crunch/keep:on
FEATURES = [nm for f in REGISTRY for nm in f.names()]
N_FEATURES = len(FEATURES)
# @crunch/keep:off
assert len(set(FEATURES)) == N_FEATURES, "duplicate feature names in REGISTRY"
print("REGISTRY: %d families, %d total features" % (len(REGISTRY), N_FEATURES))


def resolve_active_families(active_families):
    """Validate a family-name subset against REGISTRY -- fails loudly on a typo
    instead of silently emitting zero columns for that family."""
    if active_families is None:
        return None
    known = {f.name for f in REGISTRY}
    unknown = set(active_families) - known
    assert not unknown, "unknown family name(s) in ACTIVE_FAMILIES: %s" % sorted(unknown)
    return set(active_families)


# The registry IS the selection now: the Phase 1 cull removed the 25 families that
# sat at or below the permutation-null cutoff, so "every registered family" and
# "every family worth extracting" are the same set. None = every family; set it to
# a subset of
#     e2, z2, tail, time, ftest, bartlett, levene, fligner, lmom, iqr,
#     ansari, mood, sigtuk, wass, energy, mmd, entropy
# only to trade accuracy for extraction speed.
# @crunch/keep:on
ACTIVE_FAMILIES = resolve_active_families(None)
# @crunch/keep:off


REGISTRY: 13 families, 70 total features


In [18]:
import math
from scipy.special import gammaln as _gammaln

# @crunch/keep:on
_LOG_2PI = math.log(2.0 * math.pi)
# @crunch/keep:off


class BOCPDVarianceFamily(FeatureFamily):
    """Bayesian online filter for ONE variance break in `e`: "no break yet" (N(0,1)) vs
    "break before step k" for every k, post-break variance ~ Inverse-Gamma(A0, B0),
    constant hazard. Closed-form Student-t updates, vectorized over k."""
    name = "bocpd"
    channels = ("e",)
    cost = "O1"          # stateful -> runs every step
    version = 1
    HAZARD = 1.0 / 500.0
    A0 = 2.0
    B0 = 1.0

    def names(self):
        return ["bocpd_logodds", "bocpd_runlen_log", "bocpd_logvar"]

    def init(self, hist):
        cap = 1024
        self.lw = np.zeros(cap)
        self.b = np.zeros(cap)
        self.k = 0
        h = self.HAZARD
        self.c_keep = -math.log1p(-h)
        self.c_new = math.log(h) - math.log1p(-h)
        self._tables(cap)

    def _tables(self, cap):
        j = np.arange(cap, dtype=np.float64)
        a = self.A0 + 0.5 * j
        self.G = _gammaln(a + 0.5) - _gammaln(a)
        self.Acur = a

    def _grow(self):
        cap = 2 * self.lw.size
        for nm in ("lw", "b"):
            arr = np.zeros(cap)
            arr[:self.k] = getattr(self, nm)[:self.k]
            setattr(self, nm, arr)
        self._tables(cap)

    def update(self, ctx):
        x = ctx.vals["e"]
        x2 = x * x
        k = self.k
        if k + 1 >= self.lw.size:
            self._grow()
        lw, b = self.lw, self.b
        if k:
            lw[:k] += self.c_keep
        lw[k] = self.c_new
        b[k] = self.B0
        k += 1
        self.k = k
        G = self.G[k - 1::-1]
        A = self.Acur[k - 1::-1]
        B = b[:k]
        logpred = G - 0.5 * (_LOG_2PI + np.log(B)) - (A + 0.5) * np.log1p(x2 / (2.0 * B))
        lw[:k] += logpred - (-0.5 * _LOG_2PI - 0.5 * x2)
        B += 0.5 * x2
        L = float(lw[:k].max())
        wts = np.exp(lw[:k] - L)
        tot = float(wts.sum())
        logodds = L + math.log(tot)
        wts /= tot
        runlen = float(np.dot(wts, np.arange(k, 0, -1, dtype=np.float64)))
        var = float(np.dot(wts, B / (A + 0.5 - 1.0)))
        return [_safe(logodds), math.log(runlen), _safe(math.log(max(var, 1e-12)))]

# @crunch/keep:on
if not any(f.name == "bocpd" for f in REGISTRY):
    REGISTRY.append(BOCPDVarianceFamily())
FEATURES = [nm for f in REGISTRY for nm in f.names()]
N_FEATURES = len(FEATURES)
# @crunch/keep:off
print("REGISTRY: %d families, %d total features" % (len(REGISTRY), N_FEATURES))  # expect 18 and 85
print("REGISTRY: %d families, %d total features" % (len(REGISTRY), N_FEATURES))  # expect 18 and 85
print(len(REGISTRY), N_FEATURES, [f.name for f in REGISTRY][-1])

REGISTRY: 14 families, 73 total features
REGISTRY: 14 families, 73 total features
14 73 bocpd


In [ ]:
# =====================================================================
#  VOLATILITY-MODEL FAMILIES -- final versions for submission.ipynb (section 1)
#  garch (9) + argarch (11) + tgarch (11) + gjr (5) = 36 features
#  All require HistoricalRef.x_series (the ordered history).
#  NOTE for the notebook: move the `from scipy.special import ...` line to the top-level
#  imports cell -- Crunch rejects imports that are not at module level.
# =====================================================================
class GarchFamily(FeatureFamily):
    """GARCH(1,1) volatility model fitted on the historical segment; features are the online
    surprise relative to the model's own conditional variance (nu^2 = eps^2 / h_t).

    Every other family compares the online window against the UNCONDITIONAL distribution of the
    history. GARCH conditions on the recent past: volatility clustering the process itself
    generates is expected, so only departures from the fitted volatility DYNAMICS register.
    Fit: MLE with variance targeting on the ordered history (HistoricalRef.x_series), then the
    filter is warmed through that history so the first online step starts at the conditional
    variance."""
    name = "garch"
    channels = ("z",)
    cost = "O1"
    version = 3
    WINS = (32, 128, 512)
    EW = (0.02,)

    @staticmethod
    def _nll(al, be, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12:
            return 1e18
        h, s = v, 0.0
        for e2 in x2:
            s += math.log(h) + e2 / h
            h = om + al * e2 + be * h
            if h <= 1e-12:
                return 1e18
        return s

    def _fit(self, x):
        v = max(float(np.var(x)) if x.size > 8 else 1.0, 1e-8)
        x2 = (np.asarray(x, dtype=np.float64) ** 2)[-600:]
        best, bp = None, (0.0, 0.0)
        for al in (0.03, 0.06, 0.10, 0.15, 0.22, 0.30):
            for be in (0.0, 0.40, 0.65, 0.80, 0.88, 0.93):
                if al + be > 0.99:
                    continue
                f = self._nll(al, be, x2, v)
                if np.isfinite(f) and (best is None or f < best):
                    best, bp = f, (al, be)
        al, be = bp
        for step in (0.04, 0.015):
            for da in (-step, 0.0, step):
                for db in (-step, 0.0, step):
                    a2, b2 = al + da, be + db
                    if a2 < 0 or b2 < 0 or a2 + b2 > 0.99:
                        continue
                    f = self._nll(a2, b2, x2, v)
                    if f < best:
                        best, al, be = f, a2, b2
        return float(v * (1.0 - al - be)), float(al), float(be), float(v)

    def names(self):
        out = []
        for W in self.WINS:
            out += ["garch_nu2_w%d" % W, "garch_gap_w%d" % W]
        out += ["garch_nu2_ew%g" % a for a in self.EW]
        return out + ["garch_logh", "garch_nu2_exp"]

    def init(self, hist):
        xs = getattr(hist, "x_series", None)
        assert xs is not None and xs.size >= 50, "HistoricalRef.x_series missing"
        self.mu = float(hist.mu)
        self.sd = float(hist.sd) if hist.sd > 0 else 1.0
        eps_h = np.asarray(xs, dtype=np.float64) - self.mu
        self.om, self.al, self.be, self.v0 = self._fit(eps_h)
        h = self.v0
        for e in eps_h:
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self.cap = max(self.WINS)
        self.nu2 = np.zeros(self.cap, dtype=np.float64)
        self.gap = np.zeros(self.cap, dtype=np.float64)
        self.n = 0
        self.s1 = 0.0
        self.ew = [0.0] * len(self.EW)

    def _win(self, a, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return a[s:i + 1] if s >= 0 else np.concatenate([a[s % self.cap:], a[:i + 1]])

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + self.al * eps * eps + self.be * h
        i = self.n % self.cap
        self.n += 1
        self.nu2[i] = (nu2 - 1.0) * 0.7071067811865476
        self.gap[i] = math.log(max(eps * eps, 1e-12) / h) if eps != 0.0 else 0.0
        self.s1 += self.nu2[i]
        out = []
        for W in self.WINS:
            a = self._win(self.nu2, W); b = self._win(self.gap, W)
            m = a.size
            if m < 8:
                out += [0.0, 0.0]
                continue
            out.append(_safe(float(a.mean()) * math.sqrt(m)))
            out.append(_safe(float(b.mean()) * math.sqrt(m)))
        for j, al in enumerate(self.EW):
            self.ew[j] += al * (self.nu2[i] - self.ew[j])
            out.append(_safe(self.ew[j] * math.sqrt(min(self.n, 2.0 / al))))
        out.append(_safe(math.log(max(self.h, 1e-12) / max(self.v0, 1e-12))))
        out.append(_safe(self.s1 / self.n * math.sqrt(self.n)))
        return out


# =====================================================================
#  Four conditional models of the HISTORY, each asking "does the online data still follow
#  the dynamics this model learned?"  All need HistoricalRef.x_series (the ordered history).
#    argarch  GARCH on the AR(3) residuals      -> joint mean + variance dynamics
#    tgarch   GARCH with Student-t innovations  -> proper (constant) kurtosis
#    gjr      GJR-GARCH                          -> asymmetric (leverage) volatility response
#    caviar   CAViaR conditional quantiles       -> dynamic tails, tested by exceedance rates
# =====================================================================

# @crunch/keep:on
_SQ2 = 0.7071067811865476
# @crunch/keep:off


def _history(hist):
    xs = getattr(hist, "x_series", None)
    assert xs is not None and xs.size >= 50, "HistoricalRef.x_series missing -- run the patch cell"
    return np.asarray(xs, dtype=np.float64)


def _ar_resid(xs, mu, ar):
    """AR(p) residuals of the centred history, using the AR coefficients fitted in HistoricalRef."""
    c = xs - mu
    p = len(ar)
    r = c.copy()
    for k in range(1, p + 1):
        r[p:] -= ar[k - 1] * c[p - k:-k]
    return r[p:]


def _garch_grid(x2, v, extra=None, nll=None):
    """Grid + local refinement over (alpha, beta[, extra]) with variance targeting."""
    best, bp = None, None
    extras = extra if extra is not None else (None,)
    for ex in extras:
        for al in (0.03, 0.08, 0.15, 0.25):
            for be in (0.0, 0.50, 0.75, 0.88, 0.94):
                f = nll(al, be, ex, x2, v)
                if np.isfinite(f) and (best is None or f < best):
                    best, bp = f, (al, be, ex)
    al, be, ex = bp
    for step in (0.04, 0.015):
        for da in (-step, 0.0, step):
            for db in (-step, 0.0, step):
                a2, b2 = al + da, be + db
                if a2 < 0 or b2 < 0:
                    continue
                f = nll(a2, b2, ex, x2, v)
                if f < best:
                    best, al, be = f, a2, b2
    return al, be, ex


class _CondBase(FeatureFamily):
    """Shared plumbing: a ring buffer per derived stream, window/EWMA/expanding aggregates."""
    channels = ("z",)
    cost = "O1"
    version = 1
    WINS = (32, 128, 512)
    STREAMS = ()          # names of the derived per-step streams, each ~ mean 0 under the null

    def _setup_buffers(self):
        self.cap = max(self.WINS)
        self.buf = {s: np.zeros(self.cap) for s in self.STREAMS}
        self.acc = {s: 0.0 for s in self.STREAMS}
        self.ew = {s: 0.0 for s in self.STREAMS}
        self.n = 0

    def _push(self, vals):
        i = self.n % self.cap
        self.n += 1
        for s, v in vals.items():
            self.buf[s][i] = v
            self.acc[s] += v
            self.ew[s] += 0.02 * (v - self.ew[s])

    def _win(self, a, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return a[s:i + 1] if s >= 0 else np.concatenate([a[s % self.cap:], a[:i + 1]])

    def _aggregate(self):
        out = []
        for s in self.STREAMS:
            for W in self.WINS:
                w = self._win(self.buf[s], W)
                out.append(_safe(float(w.mean()) * math.sqrt(w.size)) if w.size >= 8 else 0.0)
            out.append(_safe(self.ew[s] * math.sqrt(min(self.n, 100.0))))
            out.append(_safe(self.acc[s] / self.n * math.sqrt(self.n)))
        return out

    def _agg_names(self):
        return [f"{self.name}_{s}_{k}" for s in self.STREAMS
                for k in [f"w{W}" for W in self.WINS] + ["ew", "exp"]]


# ---------------------------------------------------------------- 1. AR-GARCH
class ARGarchFamily(_CondBase):
    """GARCH on the AR(3) residuals instead of the raw series: the mean dynamics are removed first,
    so the volatility model sees genuine innovations. Streams: nu^2 - 1 (scale surprise) and
    log(eps^2/h) (realized vs expected variance)."""
    name = "argarch"
    STREAMS = ("nu2", "gap")

    @staticmethod
    def _nll(al, be, _ex, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12 or al + be >= 0.99:
            return 1e18
        h, s = v, 0.0
        for e2 in x2:
            s += math.log(h) + e2 / h
            h = om + al * e2 + be * h
        return s

    def names(self):
        return self._agg_names() + [f"{self.name}_logh"]

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        self.ar = [float(a) for a in hist.ar_l]
        r = _ar_resid(xs, self.mu, self.ar)
        v = max(float(np.var(r)), 1e-10)
        al, be, _ = _garch_grid((r ** 2)[-600:], v, nll=self._nll)
        self.om, self.al, self.be, self.v0 = v * (1 - al - be), al, be, v
        h = v
        for e in r:
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self.lags = list((xs[-len(self.ar):] - self.mu)[::-1])    # most recent first
        self._setup_buffers()

    def update(self, ctx):
        c = float(ctx.zr) * self.sd
        eps = c - sum(a * l for a, l in zip(self.ar, self.lags))
        self.lags = [c] + self.lags[:-1]
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + self.al * eps * eps + self.be * h
        self._push({"nu2": (nu2 - 1.0) * _SQ2,
                    "gap": math.log(max(eps * eps, 1e-12) / h)})
        return self._aggregate() + [_safe(math.log(self.h / self.v0))]


# ---------------------------------------------------------------- 2. Student-t GARCH
class TGarchFamily(_CondBase):
    """GARCH with Student-t innovations: the degrees of freedom are estimated with alpha/beta, so the
    innovations are standardized by the RIGHT distribution. Each standardized innovation is mapped
    through the t CDF and back to a normal score, so under the null the stream is exactly N(0,1)
    even for fat-tailed data. Streams: z^2 - 1 (scale) and 1[|z|>2] - P(|Z|>2) (tail rate)."""
    name = "tgarch"
    STREAMS = ("z2", "tail2")
    NUS = (4.0, 6.0, 10.0, 30.0)
    P2 = 0.04550026389635842

    @staticmethod
    def _nll(al, be, nu, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12 or al + be >= 0.99:
            return 1e18
        c = _gln((nu + 1) / 2) - _gln(nu / 2) - 0.5 * math.log(math.pi * (nu - 2))
        h, s = v, 0.0
        for e2 in x2:
            s -= c - 0.5 * math.log(h) - (nu + 1) / 2 * math.log1p(e2 / ((nu - 2) * h))
            h = om + al * e2 + be * h
        return s

    def names(self):
        return self._agg_names() + [f"{self.name}_nu"]

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        eps = xs - self.mu
        v = max(float(np.var(eps)), 1e-10)
        al, be, nu = _garch_grid((eps ** 2)[-600:], v, extra=self.NUS, nll=self._nll)
        self.om, self.al, self.be, self.nu, self.v0 = v * (1 - al - be), al, be, nu, v
        self.tscale = math.sqrt(nu / (nu - 2.0))
        h = v
        for e in eps:
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self._setup_buffers()

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        s = eps / math.sqrt(h) * self.tscale
        u = min(max(float(_stdtr(self.nu, s)), 1e-12), 1 - 1e-12)
        z = float(_ndtri(u))
        self.h = self.om + self.al * eps * eps + self.be * h
        self._push({"z2": (z * z - 1.0) * _SQ2,
                    "tail2": (1.0 if abs(z) > 2.0 else 0.0) - self.P2})
        return self._aggregate() + [_safe(math.log(self.nu))]




# ---------------------------------------------------------------- 3. GJR-GARCH (v2)
class GJRGarchFamily(_CondBase):
    """GJR-GARCH: negative shocks can raise volatility more than positive ones
    (h_t = omega + (alpha + gamma*1[eps<0]) eps^2 + beta h). gamma shapes the filter but is not
    emitted; the asymmetry stream carried no signal. Stream: nu^2 - 1, the scale surprise relative
    to the asymmetric volatility model."""
    name = "gjr"
    version = 2
    STREAMS = ("nu2",)
    GAMMAS = (0.0, 0.05, 0.12, 0.25)

    @staticmethod
    def _nll(al, be, ga, x, v):
        om = v * (1.0 - al - ga / 2.0 - be)
        if om <= 1e-12 or al + ga / 2.0 + be >= 0.99:
            return 1e18
        h, s = v, 0.0
        for e in x:
            e2 = e * e
            s += math.log(h) + e2 / h
            h = om + (al + (ga if e < 0 else 0.0)) * e2 + be * h
        return s

    def names(self):
        return self._agg_names()

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        eps = xs - self.mu
        v = max(float(np.var(eps)), 1e-10)
        al, be, ga = _garch_grid(eps[-600:], v, extra=self.GAMMAS, nll=self._nll)
        self.al, self.be, self.ga, self.v0 = al, be, ga, v
        self.om = v * (1 - al - ga / 2 - be)
        h = v
        for e in eps:
            h = self.om + (al + (ga if e < 0 else 0.0)) * e * e + be * h
        self.h = max(h, 1e-12)
        self._setup_buffers()

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + (self.al + (self.ga if eps < 0 else 0.0)) * eps * eps + self.be * h
        self._push({"nu2": (nu2 - 1.0) * _SQ2})
        return self._aggregate()


# @crunch/keep:on
for _cls in (GarchFamily, ARGarchFamily, TGarchFamily, GJRGarchFamily):
    if not any(f.name == _cls.name for f in REGISTRY):
        REGISTRY.append(_cls())
FEATURES = [nm for f in REGISTRY for nm in f.names()]
N_FEATURES = len(FEATURES)
# @crunch/keep:off
print("REGISTRY: %d families, %d total features" % (len(REGISTRY), N_FEATURES))   # 18 and 109


## 2. Building the training matrix

One row per `(series, online step)`. With 10k series x up to 1000 steps that is up to 10M rows,
so we subsample steps on a geometric schedule (dense early, sparse late) and carry an
**inverse-sampling-probability weight** so the fitted probability stays calibrated.

Validation series are kept at **full resolution** — TS-AUC must be evaluated on every step.

`extract_series`/`build_dataset` now take an optional `active_families` override (defaults to
the global `ACTIVE_FAMILIES`). The offline iteration loop (diagnostic, cache, CV/ablate) always
calls with `active_families=None` (full registry) regardless of the global -- see section 2b --
so trying a feature subset is a column-selection problem on an already-cached matrix, never a
re-extraction. The crunch-submission `train()`/`infer()` (sections 4/5) leave it at the global
`ACTIVE_FAMILIES`, so the deployed model only ever pays for the families it actually uses.


### Extraction backend

`build_dataset(backend=)` used to be accepted and then ignored -- the call hard-coded
`backend="loky"`. It is honored now, defaulting to the module-level `EXTRACT_BACKEND`.

**Why there is a fallback chain.** loky spawns fresh interpreters, and every worker
re-imports `__main__`. Under `crunch_tools.test()` that import resolves to a path which
does not exist in this environment, and the workers die before running any of our code:

```
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/main.py'
```

That is crunch's process-initializer monkey-patch, not a bug in the extractor, and it
is not something this notebook can fix from the inside. So `EXTRACT_BACKEND = "auto"`:

1. **loky** -- the offline default. Real process parallelism, and it survives the
   notebook namespace because joblib pickles with cloudpickle. Skipped up front when
   `__main__.__file__` points at a file that is not on disk, which is exactly the
   crunch-runner situation, so we don't pay for a spawn that is going to fail.
2. **fork `Pool`** (`multiprocessing.get_context("fork")`, `chunksize=8`) -- Linux and
   macOS. Workers inherit the interpreter image instead of re-importing `__main__`, so
   crunch's patched initializer never runs. Offered only when `fork` is in
   `multiprocessing.get_all_start_methods()`.
3. **serial** -- always works.

Naming a backend explicitly (`build_dataset(..., backend="fork")`) honors it with **no**
fallback: when you are diagnosing a backend you want the failure, not a silent downgrade.

Two things this needed. `_extract_one` is module-level rather than a closure inside
`build_dataset`, because `multiprocessing.Pool` pickles the callable by reference and a
closure cannot be pickled at all. And the `_USE_GLOBAL_ACTIVE` sentinel is resolved in
the parent before dispatch: it is a bare `object()`, so a pickled copy is a *different*
object in the worker, the `is` check in `extract_series` fails, and the sentinel gets
read as a set of family names. Only `None` or a real set crosses a process boundary now.


In [19]:
# @crunch/keep:on
KEEP_EVERY = ((24, 1), (128, 2), (512, 4), (10 ** 9, 8))   # (up_to_t, stride)
# @crunch/keep:off

import functools
import multiprocessing
import sys

# @crunch/keep:on
_USE_GLOBAL_ACTIVE = object()   # sentinel: "use whatever ACTIVE_FAMILIES is right now"
# @crunch/keep:off

# "auto" | "loky" | "fork" | "threading" | "serial"
# "auto" is the offline default: try loky, fall back to a fork Pool, then to serial.
# Naming one explicitly honors it with NO fallback, which is what you want when you
# are diagnosing a backend rather than getting work done.
# @crunch/keep:on
EXTRACT_BACKEND = "auto"
FORK_CHUNKSIZE = 8
# @crunch/keep:off


def _stride_at(t: int) -> int:
    for up_to, stride in KEEP_EVERY:
        if t < up_to:
            return stride
    return KEEP_EVERY[-1][1]


def extract_series(x_hist, x_online, tau=None, subsample=True, active_families=_USE_GLOBAL_ACTIVE):
    """Run the extractor over one series. Returns (X, y, tvec, w).

    active_families: _USE_GLOBAL_ACTIVE (default) resolves to the current
    ACTIVE_FAMILIES global at call time (this is what train()/infer() want --
    fast, submission-path extraction). Pass None explicitly to force the FULL
    registry regardless of the global (this is what the offline cache/diagnostic/
    ablate loop wants -- always extract everything, select columns after)."""
    fams = ACTIVE_FAMILIES if active_families is _USE_GLOBAL_ACTIVE else active_families
    fx = StreamFeatures(x_hist, active_families=fams)
    rows, ys, ts, ws = [], [], [], []
    for t, point in enumerate(x_online):
        f = fx.update(point)
        if subsample:
            stride = _stride_at(t)
            if t % stride != 0:
                continue
            ws.append(float(stride))
        else:
            ws.append(1.0)
        rows.append(list(f))
        ys.append(0 if (tau is None or t < tau) else 1)
        ts.append(t)
    X = np.asarray(rows, dtype=np.float32)
    return X, np.asarray(ys, np.int8), np.asarray(ts, np.int32), np.asarray(ws, np.float32)


def _extract_one(rec, subsample=True, active_families=None):
    """Module-level on purpose: multiprocessing.Pool pickles the callable BY REFERENCE,
    and a closure defined inside build_dataset cannot be pickled at all."""
    did, xh, xo, tau = rec
    X, y, t, w = extract_series(xh, xo, tau, subsample=subsample,
                                 active_families=active_families)
    g = np.full(t.shape, did, dtype=np.int64)
    return X, y, t, w, g


def _loky_can_start() -> bool:
    """loky spawns fresh interpreters that re-import __main__. crunch_tools.test()
    installs a process initializer pointing __main__ at a module path that does not
    exist in this environment, and every worker then dies with
    FileNotFoundError: /kaggle/working/main.py before running any of our code.

    If __main__ claims a __file__ that isn't on disk, spawning is going to fail, so
    don't pay for the attempt."""
    main = sys.modules.get("__main__")
    path = getattr(main, "__file__", None)
    return not (path is not None and not os.path.exists(path))


def _backend_chain(backend):
    if backend != "auto":
        return [backend]                       # explicit request: no silent fallback
    chain = []
    if _loky_can_start():
        chain.append("loky")
    if "fork" in multiprocessing.get_all_start_methods():
        chain.append("fork")
    chain.append("serial")
    return chain


def _run_fork(fn, items, n_jobs):
    ctx = multiprocessing.get_context("fork")
    procs = None if n_jobs in (-1, 0) else n_jobs
    with ctx.Pool(processes=procs) as pool:
        return pool.map(fn, items, chunksize=FORK_CHUNKSIZE)


def _stack(res):
    return (np.concatenate([r[0] for r in res]),
            np.concatenate([r[1] for r in res]),
            np.concatenate([r[2] for r in res]),
            np.concatenate([r[3] for r in res]),
            np.concatenate([r[4] for r in res]))


def build_dataset(datasets, subsample=True, max_series=None, n_jobs=-1, verbose=1,
                   active_families=_USE_GLOBAL_ACTIVE, backend=None):
    """Extract many series in parallel. `backend` overrides EXTRACT_BACKEND for this
    call; None means "use the global".

    Each family is .clone()-d off the REGISTRY prototype (see FeatureFamily.clone()),
    so concurrent extraction never shares mutable per-series state.
    """
    items = list(datasets)[: max_series] if max_series else list(datasets)
    if not items:
        raise ValueError("build_dataset got no series")

    # Resolve the sentinel HERE, in the parent. It is a bare object(); pickled into a
    # worker it becomes a different object, the `is` test in extract_series fails, and
    # the sentinel gets read as a set of family names.
    fams = ACTIVE_FAMILIES if active_families is _USE_GLOBAL_ACTIVE else active_families
    worker = functools.partial(_extract_one, subsample=subsample, active_families=fams)

    if n_jobs == 1 or len(items) == 1:
        return _stack([worker(r) for r in items])

    chain = _backend_chain(backend or EXTRACT_BACKEND)
    errors = []
    for b in chain:
        try:
            if b == "serial":
                if verbose:
                    print("build_dataset: serial over %d series" % len(items))
                return _stack([worker(r) for r in items])
            if b == "fork":
                if verbose:
                    print("build_dataset: fork Pool over %d series (chunksize %d)" % (len(items), FORK_CHUNKSIZE))
                return _stack(_run_fork(worker, items, n_jobs))
            res = joblib.Parallel(n_jobs=n_jobs, verbose=verbose, batch_size=8, backend=b)(
                joblib.delayed(worker)(r) for r in items)
            return _stack(res)
        except Exception as exc:
            errors.append((b, type(exc).__name__, str(exc)[:200]))
            if b == chain[-1]:
                raise RuntimeError(
                    "every extraction backend failed: %s" % "; ".join("%s -> %s: %s" % e for e in errors)) from exc
            print("build_dataset: backend %r failed (%s: %s) -- falling back to %r" % (b, type(exc).__name__, str(exc)[:160], chain[chain.index(b) + 1]))


## 2b. Per-family disk cache

One `.npy` per family per sample, not one blob per feature list:

```
cache/<sample_key>/meta.joblib                  y, t, w, g, series_ids, KEEP_EVERY, seed
cache/<sample_key>/fam__<name>__<fam_key>.npy   float32 (n_rows, len(names()))
cache/<sample_key>/null.joblib                  CUTOFF + null table, per feature set
```

`sample_key` identifies the **row set** (sorted series ids + `KEEP_EVERY` + subsample
flag + seed). `family_key` identifies one family's **output**: its `.name`, its
`names()`, a manual `version` class attribute, `inspect.getsource(type(fam))`, and the
extractor constants it reads (`WINDOWS`, `TWOSAMPLE_CHANNELS`, `ALPHAS`, `DRIFTS`,
`TAIL_W`, `BIN_W`, `NBINS`, `AR_P`, `RefSub.CAP`, `REF_MAX`).

This is what makes adding a feature cheap. The old cache keyed on the whole feature
list, so one new family invalidated the lot and re-extracted all 17. Now
`get_cached_matrix` loads every family that exists and calls `build_dataset` with
`active_families={the missing ones}` only.

**`version` is not optional bookkeeping.** `inspect.getsource` has no file to read for
a class defined in a notebook cell, so in Jupyter it usually raises and the source
falls back to `""` silently. When that happens the *only* thing separating v1 of a
family from v2 is `version`, so bump it by hand after editing a family body or you
will get the old columns back without a warning.

Rows are checked, not assumed: whenever a later call extracts a new family into an
existing sample, the fresh `y`/`t`/`g` are asserted equal to `meta.joblib`'s before
any column is spliced in.

`clear_cache()` wipes the directory; `clear_cache("mmd")` drops one family everywhere.
`cache_status(sample)` prints what is on disk. `FEATURE_TO_FAMILY` is now built by
`feature_to_family()` **at call time** -- the frozen module-level dict missed any
family registered after this cell ran, which is exactly what section 3d does.


In [20]:
import fnmatch
import glob
import hashlib
import inspect
import json
import shutil

# @crunch/keep:on
CACHE_DIR = "cache"
# @crunch/keep:off


def feature_to_family():
    """Exact feature -> family-name map, built from REGISTRY at CALL time.

    Rebuilt on every call on purpose: `research_step` (section 3d) adds and removes
    families from REGISTRY between calls, and a dict frozen when this cell ran would
    silently miss every family registered afterwards.

    Built from REGISTRY, not guessed from the feature string -- string-splitting
    breaks for any family whose .name itself contains an underscore. The archived
    "perm_ent" family is the worked example: "perm_ent_z_w16" would split to "perm".
    """
    return {nm: f.name for f in REGISTRY for nm in f.names()}


# @crunch/keep:on
FEATURE_TO_FAMILY = feature_to_family()   # convenience snapshot for interactive use;
# @crunch/keep:off
                                          # every function below calls the function


# =====================================================================
#  Cache keys
# =====================================================================
def _short_hash(payload) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def _extractor_config():
    """The module-level constants the extracted numbers depend on. Changing any one of
    these changes what every family emits, so all of them go into every family key."""
    return {
        "WINDOWS": list(WINDOWS),
        "TWOSAMPLE_CHANNELS": list(TWOSAMPLE_CHANNELS),
        "ALPHAS": list(ALPHAS),
        "DRIFTS": list(DRIFTS),
        "TAIL_W": TAIL_W,
        "BIN_W": BIN_W,
        "NBINS": NBINS,
        "AR_P": AR_P,
        "REFSUB_CAP": RefSub.CAP,
        "REF_MAX": REF_MAX,
    }


def sample_key(series_ids, subsample=True, seed=0) -> str:
    """Identity of the ROW SET: which series, at what subsample schedule, which seed."""
    return _short_hash({
        "series_ids": sorted(str(s) for s in series_ids),
        "schedule": [list(p) for p in KEEP_EVERY] if subsample else [],
        "subsample": bool(subsample),
        "seed": seed,
    })


def family_key(fam) -> str:
    """Identity of one family's OUTPUT: name, column names, a manual version counter,
    its source, and the extractor config it reads.

    inspect.getsource() has no file to read for a class defined in a notebook cell and
    raises there; the fallback is silent, which is exactly why `version` exists. Bump
    `version` by hand after editing a family body, or the stale columns are reused.
    """
    try:
        source = inspect.getsource(type(fam))
    except Exception:
        source = ""
    return _short_hash({
        "name": fam.name,
        "names": list(fam.names()),
        "version": int(getattr(fam, "version", 1)),
        "source": source,
        "config": _extractor_config(),
    })


def _sample_dir(skey, cache_dir=CACHE_DIR):
    return os.path.join(cache_dir, skey)


def _fam_path(d, fam):
    return os.path.join(d, "fam__%s__%s.npy" % (fam.name, family_key(fam)))


# =====================================================================
#  The store
# =====================================================================
def get_cached_matrix(datasets, subsample=True, max_series=None, seed=0,
                       cache_dir=CACHE_DIR, families=None, verbose=True):
    """Assemble (X, y, t, w, g, feature_names) for `datasets`, extracting only the
    families that are not already on disk.

    Same return signature as before, and the columns come back in REGISTRY order, so
    quick_train / ablate / cv_evaluate / select_features need no change.

    `families=None` means every registered family; pass a set of names to assemble a
    subset (it still only extracts what is missing from that subset).
    """
    global FEATURE_TO_FAMILY
    FEATURE_TO_FAMILY = feature_to_family()

    items = list(datasets)[:max_series] if max_series else list(datasets)
    series_ids = [r[0] for r in items]
    skey = sample_key(series_ids, subsample, seed)
    d = _sample_dir(skey, cache_dir)
    os.makedirs(d, exist_ok=True)

    wanted = [f for f in REGISTRY if families is None or f.name in families]
    meta_path = os.path.join(d, "meta.joblib")
    meta = joblib.load(meta_path) if os.path.exists(meta_path) else None

    have, missing = {}, []
    for f in wanted:
        p = _fam_path(d, f)
        if meta is not None and os.path.exists(p):
            have[f.name] = np.load(p)
        else:
            missing.append(f)

    if missing:
        if verbose:
            print("cache %s | extracting %d of %d families over %d series: %s" % (skey, len(missing), len(wanted), len(items),
                     ", ".join(f.name for f in missing)))
        Xm, y, t, w, g = build_dataset(items, subsample=subsample,
                                        active_families={f.name for f in missing},
                                        verbose=0)
        if meta is None:
            meta = {"y": y, "t": t, "w": w, "g": g,
                    "series_ids": [str(s) for s in series_ids],
                    "KEEP_EVERY": [list(p) for p in KEEP_EVERY],
                    "subsample": bool(subsample), "seed": seed}
            joblib.dump(meta, meta_path)
        else:
            # A later extraction has to line up row-for-row with the rows the cached
            # families were built against. If it does not, the assembled matrix would
            # silently splice columns from two different row sets together.
            assert np.array_equal(y, meta["y"]), "fresh y != meta.joblib y"
            assert np.array_equal(t, meta["t"]), "fresh t != meta.joblib t"
            assert np.array_equal(g, meta["g"]), "fresh g != meta.joblib g"

        off = 0
        for f in missing:
            k = len(f.names())
            block = np.ascontiguousarray(Xm[:, off:off + k], dtype=np.float32)
            off += k
            for stale in glob.glob(os.path.join(d, "fam__%s__*.npy" % f.name)):
                os.remove(stale)          # a previous version of this family's columns
            np.save(_fam_path(d, f), block)
            have[f.name] = block
        assert off == Xm.shape[1], (
            "column accounting: consumed %d of %d extracted columns" % (off, Xm.shape[1]))
    elif verbose:
        print("cache %s | all %d families cached (%d series, %d rows)" % (skey, len(wanted), len(items), len(meta["y"])))

    names = []
    blocks = []
    for f in wanted:
        blocks.append(have[f.name])
        names.extend(f.names())
    X = (np.concatenate(blocks, axis=1) if blocks
         else np.empty((len(meta["y"]), 0), dtype=np.float32))
    return X, meta["y"], meta["t"], meta["w"], meta["g"], names


def clear_cache(family=None, cache_dir=CACHE_DIR):
    """clear_cache() wipes everything; clear_cache("mmd") drops one family's columns
    from every sample, leaving meta/null and the other families intact."""
    if not os.path.isdir(cache_dir):
        print("no cache directory at", cache_dir)
        return
    if family is None:
        shutil.rmtree(cache_dir)
        print("cache cleared:", cache_dir)
        return
    n = 0
    for skey in sorted(os.listdir(cache_dir)):
        for p in glob.glob(os.path.join(cache_dir, skey, "fam__%s__*.npy" % family)):
            os.remove(p)
            n += 1
    print("cleared %d cached file(s) for family %r" % (n, family))


def cache_status(datasets, subsample=True, max_series=None, seed=0, cache_dir=CACHE_DIR):
    """What is on disk for this sample, family by family."""
    items = list(datasets)[:max_series] if max_series else list(datasets)
    skey = sample_key([r[0] for r in items], subsample, seed)
    d = _sample_dir(skey, cache_dir)
    print("sample %s (%d series) -> %s" % (skey, len(items), d))
    for f in REGISTRY:
        p = _fam_path(d, f)
        print("   %-10s %-8s %s" % (f.name, "CACHED" if os.path.exists(p) else "missing",
                                     os.path.basename(p)))
    return skey


# ---- permutation-null side-cache, keyed by the feature set it was computed over ----
def _null_path(skey, cache_dir=CACHE_DIR):
    return os.path.join(_sample_dir(skey, cache_dir), "null.joblib")


def load_null(skey, key, cache_dir=CACHE_DIR):
    p = _null_path(skey, cache_dir)
    if not os.path.exists(p):
        return None
    return joblib.load(p).get(key)


def save_null(skey, key, value, cache_dir=CACHE_DIR):
    p = _null_path(skey, cache_dir)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    store = joblib.load(p) if os.path.exists(p) else {}
    store[key] = value
    joblib.dump(store, p)


def select_features(feature_names, families=None, features=None, patterns=None,
                     min_ts_auc=None, diag=None):
    """Resolve any combination of family names / explicit feature names / glob
    patterns / a diagnostic-driven min_ts_auc rule into column indices. Returns
    (idx, matching_names). No selector given -> every column."""
    feature_names = list(feature_names)
    f2f = feature_to_family()
    chosen = set()
    any_selector = False
    if families:
        any_selector = True
        fams = set(families)
        chosen |= {i for i, nm in enumerate(feature_names) if f2f.get(nm) in fams}
    if features:
        any_selector = True
        want = set(features)
        chosen |= {i for i, nm in enumerate(feature_names) if nm in want}
    if patterns:
        any_selector = True
        for pat in patterns:
            chosen |= {i for i, nm in enumerate(feature_names) if fnmatch.fnmatch(nm, pat)}
    if min_ts_auc is not None:
        any_selector = True
        assert diag is not None, "min_ts_auc requires diag= a diagnostic DataFrame (section 3)"
        keep = set(diag.loc[diag["ts_auc_abs"] >= min_ts_auc, "feature"])
        chosen |= {i for i, nm in enumerate(feature_names) if nm in keep}
    if not any_selector:
        chosen = set(range(len(feature_names)))
    idx = np.array(sorted(chosen), dtype=np.int64)
    return idx, [feature_names[i] for i in idx]


## 3. TS-AUC, and a per-feature diagnostic

`ts_auc` is the leaderboard metric. Same number as the sklearn `roc_auc_score`
loop it replaces, computed from within-`t` ranks instead: inside a `t`-slice,
`AUC = (sum of positive ranks - P(P+1)/2) / (P*N)`. `TSRanker` does the grouping by
`t` once and `run_feature_diagnostic` shares one instance between the results table
and the permutation null.

That sharing is what makes the null cheap: **the within-`t` ranks of a feature do not
depend on the labels**, so each feature is ranked once and every permutation is two
`np.bincount` calls over those same ranks. The permutation scheme, the RNG stream and
the output are identical to the previous version.

`run_feature_diagnostic` replaces `single_feature_ts_auc`: it samples series **uniformly at
random** (seeded, default 10k) rather than taking the first N, extracts the **full registry**
through the section 2b cache (so re-running after adding one family doesn't re-extract
everything else), and folds each feature's TS-AUC via `max(a, 1-a)` as before -- but now next to
an honest noise floor from `permutation_null_ts_auc`, because the raw folded number cannot be
thresholded on its own. Two reasons:

1. **Folding bias.** `max(a, 1-a)` does not center a genuinely null feature on 0.50; sampling
   noise pushes it up in both tails and only the upper one survives the fold, so null features
   look positive on average.
2. **Effective n is series count, not row count.** The same series reappears in every `t`-slice,
   so per-slice AUCs for one series are strongly correlated -- hundreds of thousands of rows
   buy far fewer independent observations than that number suggests.

`permutation_null_ts_auc` shuffles **series-level** break labels (which series get treated as
breaking), preserving each series' own row/`t` structure -- never shuffling rows independently,
which would destroy exactly the within-series correlation that inflates the naive null. It
reports the mean and the 95th percentile of the folded AUC under that null; the P95 is printed
as `CUTOFF` and used as the `above_null` column in the results table.


In [21]:
class TSRanker:
    """Groups rows by `t` once, so a TS-AUC over that row set costs two bincounts.

    Within a t-slice, AUC = (sum of positive ranks - P(P+1)/2) / (P*N). The within-t
    ranks of a feature do not depend on the labels at all, so the permutation null can
    rank each feature ONCE and then re-score every permutation from the same ranks --
    which is what turns an O(n_perm * n_features * n log n) loop into O(n_features *
    n log n) plus O(n_perm * n_features * n).

    Build one per row set and share it: `run_feature_diagnostic` hands the same
    instance to `_folded_ts_auc_table` and `permutation_null_ts_auc`."""

    def __init__(self, t):
        t = np.asarray(t)
        self.order_t = np.argsort(t, kind="mergesort")
        ts = t[self.order_t]
        _, self.gid_sorted = np.unique(ts, return_inverse=True)
        self.gid = np.empty_like(self.gid_sorted)
        self.gid[self.order_t] = self.gid_sorted          # group id per original row
        self.n_groups = int(self.gid_sorted.max()) + 1 if ts.size else 0

    def ranks(self, score):
        """Average within-t ranks (1-based, ties averaged) for every row, original order."""
        s = np.asarray(score, dtype=np.float64)
        s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
        o = np.lexsort((s, self.gid))                     # sort by group, then score
        g_o, s_o = self.gid[o], s[o]
        n = s.size
        new_run = np.empty(n, dtype=bool)
        new_run[0] = True
        new_run[1:] = (g_o[1:] != g_o[:-1]) | (s_o[1:] != s_o[:-1])
        new_grp = np.empty(n, dtype=bool)
        new_grp[0] = True
        new_grp[1:] = g_o[1:] != g_o[:-1]
        pos = np.arange(n, dtype=np.float64)
        grp_start = np.maximum.accumulate(np.where(new_grp, pos, 0.0))
        run_id = np.cumsum(new_run) - 1
        run_start = pos[new_run]
        run_len = np.diff(np.append(run_start, n))
        avg = (run_start - grp_start[new_run]) + (run_len + 1) / 2.0
        r = np.empty(n, dtype=np.float64)
        r[o] = avg[run_id]
        return r

    def auc_from_ranks(self, y, r):
        """Row-pair-weighted mean of the per-t AUCs, from precomputed ranks."""
        y = np.asarray(y).astype(np.float64)
        G = self.n_groups
        npos = np.bincount(self.gid, weights=y, minlength=G)
        ntot = np.bincount(self.gid, minlength=G).astype(np.float64)
        nneg = ntot - npos
        rsum = np.bincount(self.gid, weights=r * y, minlength=G)
        ok = (npos > 0) & (nneg > 0)
        if not ok.any():
            return 0.5
        w = npos[ok] * nneg[ok]
        auc = (rsum[ok] - npos[ok] * (npos[ok] + 1) / 2.0) / w
        return float((w * auc).sum() / w.sum())


def ts_auc(y, t, score):
    """The leaderboard metric: AUC within each t-slice, weighted by that slice's
    positive-negative pair count. Same value as the sklearn roc_auc_score loop it
    replaces, without the per-slice Python call."""
    rk = TSRanker(t)
    return rk.auc_from_ranks(y, rk.ranks(score))


def _folded_ts_auc_table(X, y, t, feature_names, ranker=None):
    rk = ranker if ranker is not None else TSRanker(t)
    rows = []
    for k, nm in enumerate(feature_names):
        a = rk.auc_from_ranks(y, rk.ranks(X[:, k]))
        rows.append((nm, max(a, 1 - a), a))
    return pd.DataFrame(rows, columns=["feature", "ts_auc_abs", "ts_auc_raw"])


def permutation_null_ts_auc(X, y, t, g, feature_idx, feature_names, n_perm=100, seed=0,
                             ranker=None):
    """Series-level label permutation: shuffle which SERIES are treated as
    breaking, keep each series' own row/t structure intact. A relabeled
    "breaker" reuses its own original first-break t as a synthetic tau (so the
    permuted series still has a well-defined post-break region); a real
    breaker relabeled non-breaking has all its rows set to y=0. Never shuffle
    rows independently -- that destroys the within-series correlation that is
    exactly why the naive (row-count) null is too tight.

    Same permutation scheme and the same RNG stream as the row-masking version it
    replaces: one rng.permutation(breaks_arr) per iteration, on an array of the same
    length. The per-series `g == sid` mask loop is gone -- the branch is now a
    vectorized np.where over rows, and the feature ranks are computed once up front
    because they do not depend on the labels."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); t = np.asarray(t); g = np.asarray(g)
    uniq_series, inv = np.unique(g, return_inverse=True)
    S = uniq_series.size
    breaks_arr = np.zeros(S, dtype=np.int64)
    np.maximum.at(breaks_arr, inv, y.astype(np.int64))

    # per-series synthetic tau = tt[len(tt) // 2] in the series' own row order
    order = np.argsort(inv, kind="stable")
    counts = np.bincount(inv, minlength=S)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    cut = t[order[starts + counts // 2]]
    synth_y = (t >= cut[inv]).astype(y.dtype)
    orig_break_row = breaks_arr[inv] == 1

    rk = ranker if ranker is not None else TSRanker(t)
    R = [rk.ranks(X[:, j]) for j in feature_idx]
    null = [[] for _ in feature_idx]
    for _ in range(n_perm):
        pb_row = rng.permutation(breaks_arr)[inv] == 1
        y_perm = np.where(pb_row, np.where(orig_break_row, y, synth_y), 0).astype(y.dtype)
        for k, r in enumerate(R):
            a = rk.auc_from_ranks(y_perm, r)
            null[k].append(max(a, 1 - a))

    rows = [(nm, float(np.mean(v)), float(np.percentile(v, 95)))
            for nm, v in zip(feature_names, null)]
    return pd.DataFrame(rows, columns=["feature", "null_mean", "null_p95"])


def run_feature_diagnostic(train_data, seed=0, n_series=10_000, n_perm=100,
                            perm_features=None, cache_dir=CACHE_DIR):
    """Sample n_series uniformly at random, extract the full registry (through
    the section 2b cache), fold TS-AUC per feature, and attach an honest
    permutation-null cutoff. Returns (results_df, family_rollup_df, cutoff,
    (X, y, t, w, g, feature_names)) -- the last element is the raw cached matrix
    for the sampled series, handed back so downstream cells (e.g. the per-feature
    AUC-by-age plot at the bottom of the notebook) don't need a second extraction."""
    items = list(train_data)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(items), size=min(n_series, len(items)), replace=False)
    sample = [items[i] for i in idx]

    X, y, t, w, g, feature_names = get_cached_matrix(sample, subsample=True, seed=seed,
                                                       cache_dir=cache_dir)
    # A column that is constant across the whole matrix has TS-AUC exactly 0.5 and
    # tells you nothing -- but it looks identical to "real feature, no signal" in the
    # results table. `iqr` scored exactly 0.500000 in an earlier run for this reason.
    # Flag it here so a degenerate extraction can never be misread as a dead family.
    _sd = X.std(axis=0)
    _const = [nm for nm, s in zip(feature_names, _sd) if not (s > 0)]
    if _const:
        print("WARNING: %d/%d column(s) are CONSTANT across the matrix (std == 0) -- "
              "their TS-AUC of 0.5 means 'degenerate', not 'no signal':" % (len(_const), len(feature_names)))
        for nm in _const:
            print("    ", nm)

    # one grouping of the rows by t, reused by the table and the null
    ranker = TSRanker(t)
    diag = _folded_ts_auc_table(X, y, t, feature_names, ranker=ranker)
    diag["n_series"] = len(sample)
    diag["n_rows"] = X.shape[0]
    diag["family"] = diag["feature"].map(feature_to_family())

    # permutation null on a representative subset (best feature per family)
    # unless the caller asks for a specific list -- running it on all ~1500
    # features is possible but slow; the family-representative set is enough
    # to set CUTOFF.
    if perm_features is None:
        rep = diag.sort_values("ts_auc_abs", ascending=False).drop_duplicates(subset="family")
        perm_features = rep["feature"].tolist()
    perm_idx = [feature_names.index(nm) for nm in perm_features]
    null_df = permutation_null_ts_auc(X, y, t, g, perm_idx, perm_features,
                                       n_perm=n_perm, seed=seed, ranker=ranker)

    cutoff = float(null_df["null_p95"].max())
    diag = diag.merge(null_df[["feature", "null_mean", "null_p95"]], on="feature", how="left")
    diag["above_null"] = diag["ts_auc_abs"] >= cutoff
    diag = diag.sort_values("ts_auc_abs", ascending=False).reset_index(drop=True)

    rollup = (diag.groupby("family")["ts_auc_abs"]
              .agg(["max", "mean"]).sort_values("max", ascending=False).reset_index())

    print("permutation-null CUTOFF (P95 over %d representative features, %d perms): %.5f" % (len(perm_features), n_perm, cutoff))
    return diag, rollup, cutoff, (X, y, t, w, g, feature_names)


### Diagnostic run: 10k series, seeded, with a permutation-null cutoff

Interpretation rules (read before touching the results table):

- **`t_step`/`t_log` are structurally 0.5 and are always kept.** A pure function of `t` is
  constant within a `t`-slice, so TS-AUC cannot see it at all -- this is expected, not a bug,
  and it says nothing about whether `t` helps through interactions (it does, via the tree
  splits LightGBM builds on it jointly with other features).
- **A column with `std == 0` across the matrix is degenerate, not merely weak.** It gets
  TS-AUC exactly 0.5 and is indistinguishable in this table from a real feature with no
  signal, so `run_feature_diagnostic` now prints a warning listing any such column. An
  exact `0.500000` in the results table with no warning means "no signal"; with a warning
  it means "this family produced a constant and needs looking at".
- **Univariate AUC cannot see complementarity.** A feature at 0.51 that is uncorrelated with the
  `e2_*` block may add real value to the model; a feature at 0.55 that is 0.95-correlated with
  `e2_ew0.08` adds nothing once `e2_ew0.08` is already in. **Final keep/drop decisions are made
  by CV ablation (section 3c), not by this table** -- this table is for triage (what's worth
  ablating) and for measuring the honest noise floor via the permutation null, not a keep/drop
  rule on its own.


In [22]:
import time


def bench_extractor(datasets, n_series=3, n_project=2500, n_workers=4):
    """Per-family extraction cost for the current driver, and what it projects to.

    `n_project` / `n_workers` are what build_dataset will actually face -- the
    projection is just mean-seconds-per-series * n_project / n_workers, so it ignores
    worker start-up and the tail effect of uneven series lengths. Treat it as a floor.
    """
    items = list(datasets)[:n_series]
    base = {f.name for f in REGISTRY if f.cost == "O1"}

    def t_of(rec, active):
        xh, xo = rec[1], rec[2]
        fx = StreamFeatures(xh, active_families=active)
        t0 = time.perf_counter()
        for v in xo:
            fx.update(v)
        return time.perf_counter() - t0

    mean_len = float(np.mean([len(r[2]) for r in items]))
    b = float(np.mean([t_of(r, base) for r in items]))
    full = float(np.mean([t_of(r, None) for r in items]))

    print("%d series, mean online length %.0f steps" % (len(items), mean_len))
    print("  O(1) baseline (%s) : %7.4f s/series" % (",".join(sorted(base)), b))
    print("  FULL registry (%d families, %d features): %7.4f s/series" % (len(REGISTRY), N_FEATURES, full))
    print()
    print("  projected for %d series on %d workers: %5.1f min  (baseline only: %.1f min)" % (n_project, n_workers, full * n_project / n_workers / 60,
             b * n_project / n_workers / 60))
    for n in (10_000, 25_000):
        print("  projected for %6d series on %d workers: %5.1f min" % (n, n_workers, full * n / n_workers / 60))

    print("\n  marginal cost per family (on top of the O(1) baseline):")
    rows = sorted(((float(np.mean([t_of(r, base | {f.name}) for r in items])) - b,
                    f.name, f.cost) for f in REGISTRY if f.cost != "O1"), reverse=True)
    tot = sum(d for d, _, _ in rows)
    for d, n, c in rows:
        print("    %-10s %-7s %7.4f s  (%4.1f%% of the non-baseline cost)" % (n, c, d, 100.0 * d / tot if tot > 0 else 0.0))
    return full


# bench_extractor(train_data)    # (research only)


## 3b. K-fold cross-validation over series

The single `train()` call above reports TS-AUC on **one** random 15% hold-out of series.
That number has real sampling noise — with a few thousand series and a handful of break
types, a single split can favor or disfavor a given feature/param change purely by luck of
which series ended up in validation.

`cv_evaluate()` below repeats the same train/validate procedure `n_folds` times, each time
holding out a *different* group of series, and reports the **mean and spread** of TS-AUC
across folds. Only trust a change (new feature, pruned family, different LGB params) if it
moves the CV mean by more than roughly the fold-to-fold std — otherwise you're chasing noise.

Cost: this is `n_folds` full training runs, so ~5x the time of a single `train()` call.
Use `max_series` to iterate fast on a subset first, then run the full thing to confirm
before you commit to a change.

**Extended for the fast iteration loop (section 3c):** `cv_evaluate` now optionally takes a
pre-extracted `X, y, t, g` (e.g. from `get_cached_matrix`) plus a `feature_idx` column
selection, in which case it skips `build_dataset` entirely and folds directly by the unique
ids already present in `g`. This is what lets `ablate()` re-run CV under a different feature
selection in seconds -- the original `datasets=`-based call (re-extracts from raw series) still
works unchanged for anything not going through the cache.


In [23]:
def _fit_and_score(Xtr, ytr, wtr, feature_names, Xva, yva, tva, num_rounds=None):
    dtr = lgb.Dataset(Xtr, label=ytr, weight=wtr, feature_name=feature_names, free_raw_data=False)
    dva = lgb.Dataset(Xva, label=yva, feature_name=feature_names, reference=dtr, free_raw_data=False)
    booster = lgb.train(
        LGB_PARAMS, dtr, num_boost_round=num_rounds or NUM_ROUNDS,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
    )
    pva = booster.predict(Xva, num_iteration=booster.best_iteration)
    return ts_auc(yva, tva, pva)


def cv_evaluate(datasets=None, n_folds: int = 5, seed: int = 0,
                 max_series=None, num_rounds=None, verbose: bool = True,
                 X=None, y=None, t=None, g=None, feature_idx=None, feature_names=None):
    """K-fold cross-validation over series.

    Two modes:
    - `datasets=` (original behavior): re-extracts each fold from raw series via
      build_dataset (train subsampled, valid at full resolution -- the fidelity-
      first path, what train() itself uses).
    - `X=, y=, t=, g=` (fast-iteration mode, section 3c): works directly off an
      already-extracted matrix (e.g. from get_cached_matrix), optionally
      restricted to `feature_idx` columns. NOTE: both "train" and "valid" rows
      here share whatever subsample schedule the cache was built with (unlike
      the datasets= path, which validates at full resolution) -- this trades a
      little fidelity for speed, which is the point of this path. Confirm a
      real decision with the datasets= path or crunch_tools.test() before
      committing to it.
    """
    if X is not None:
        assert y is not None and t is not None and g is not None, \
            "matrix mode requires X, y, t, g"
        names = list(feature_names) if feature_names is not None else list(FEATURES)
        if feature_idx is not None:
            Xsel = X[:, feature_idx]
            names = [names[i] for i in feature_idx]
        else:
            Xsel = X

        uniq = np.unique(g)
        if max_series:
            uniq = uniq[:max_series]
            mask = np.isin(g, uniq)
            Xsel, y, t, g = Xsel[mask], y[mask], t[mask], g[mask]
            uniq = np.unique(g)

        rng = np.random.default_rng(seed)
        perm = rng.permutation(len(uniq))
        folds = np.array_split(perm, n_folds)

        fold_aucs = []
        for k in range(n_folds):
            va_ids = set(uniq[folds[k]].tolist())
            va_mask = np.isin(g, list(va_ids))
            tr_mask = ~va_mask
            auc = _fit_and_score(Xsel[tr_mask], y[tr_mask], np.ones(tr_mask.sum(), dtype=np.float32),
                                  names, Xsel[va_mask], y[va_mask], t[va_mask], num_rounds)
            fold_aucs.append(auc)
            if verbose:
                print("fold %d/%d | train %5d rows | valid %5d rows | TS-AUC %.5f" % (k + 1, n_folds, int(tr_mask.sum()), int(va_mask.sum()), auc))
    else:
        items = list(datasets)
        if max_series:
            items = items[:max_series]

        n = len(items)
        rng = np.random.default_rng(seed)
        perm = rng.permutation(n)
        folds = np.array_split(perm, n_folds)

        fold_aucs = []
        for k in range(n_folds):
            va_idx = set(folds[k].tolist())
            tr_items = [items[i] for i in range(n) if i not in va_idx]
            va_items = [items[i] for i in folds[k]]

            Xtr, ytr, ttr, wtr, _ = build_dataset(tr_items, subsample=True, verbose=0)
            Xva, yva, tva, _, _ = build_dataset(va_items, subsample=False, verbose=0)

            auc = _fit_and_score(Xtr, ytr, wtr, FEATURES, Xva, yva, tva, num_rounds)
            fold_aucs.append(auc)
            if verbose:
                print("fold %d/%d | train %4d series | valid %4d series | TS-AUC %.5f" % (k + 1, n_folds, len(tr_items), len(va_items), auc))

    fold_aucs = np.array(fold_aucs)
    if verbose:
        print("\nCV TS-AUC: %.5f +/- %.5f  (min %.5f, max %.5f)" % (fold_aucs.mean(), fold_aucs.std(), fold_aucs.min(), fold_aucs.max()))
    return fold_aucs


## 3c. The fast iteration loop: `quick_train` + `ablate`

This is the actual point of the refactor. The intended workflow is **add a feature -> test it
alone (section 3, diagnostic) -> train on a subset of features -> repeat**, and it should cost
seconds per iteration, not a re-extraction.

- `quick_train(cache, selection, ...)`: a thin wrapper over the same fit logic as `train()`
  (same `LGB_PARAMS`, same series-level split), but takes a cached matrix + column selection
  directly. No re-extraction, no disk writes, no model artifact -- just a hold-out TS-AUC number.
- `ablate(cache, baseline_selection, ...)`: given a baseline feature selection, drops (or adds)
  each candidate family in turn and re-runs `cv_evaluate` (matrix mode) on the cached matrix,
  returning `delta_mean`/`delta_std` per family vs. the baseline CV. **This is the tool that
  makes keep/drop decisions** -- it's the only one of these that can see complementarity
  (section 3's diagnostic can't: a 0.51 feature uncorrelated with the `e2_*` block may help even
  though a 0.55 feature 0.95-correlated with it won't).

**Accept-rule:** only accept a change (new feature, dropped family, different LGB params) if it
moves the **CV mean** by more than the **fold-to-fold std**. A single train()/quick_train split
difference of 0.002-0.005 is noise, not signal -- and `y_test.reduced.parquet` (section 6) is
about 1% the size of train, so it's a smoke test for the submission plumbing, never the metric
you make a keep/drop call on.


In [24]:
def quick_train(cache, selection, max_series=None, num_rounds=None, seed=0, valid_fraction=None):
    """Thin wrapper over the same fit logic as train(), on an already-cached
    matrix + column selection. No re-extraction, no disk writes, no artifact --
    just a hold-out TS-AUC. `selection` is either an index array or a
    (idx, names) tuple, e.g. straight from select_features(...)."""
    X, y, t, w, g, feature_names = cache
    idx = selection[0] if isinstance(selection, tuple) else np.asarray(selection)
    names = [feature_names[i] for i in idx]
    Xsel = X[:, idx]

    uniq = np.unique(g)
    if max_series:
        uniq = uniq[:max_series]
        mask = np.isin(g, uniq)
        Xsel, y, t, w, g = Xsel[mask], y[mask], t[mask], w[mask], g[mask]
        uniq = np.unique(g)

    vf = valid_fraction if valid_fraction is not None else VALID_FRACTION
    rng = np.random.default_rng(seed)
    n_val = max(1, int(round(len(uniq) * vf)))
    val_ids = set(rng.choice(uniq, size=n_val, replace=False).tolist())
    va_mask = np.isin(g, list(val_ids))
    tr_mask = ~va_mask

    return _fit_and_score(Xsel[tr_mask], y[tr_mask], w[tr_mask], names,
                           Xsel[va_mask], y[va_mask], t[va_mask], num_rounds)


def ablate(cache, baseline_idx, candidates="families", n_folds=5, seed=0,
           num_rounds=None, max_series=None, verbose=True):
    """For each candidate family, try dropping it (if in the baseline selection)
    or adding it (if not), re-run cv_evaluate on the cached matrix, and report
    the delta vs. the baseline CV mean. This is the tool that makes keep/drop
    decisions -- it sees complementarity that the univariate diagnostic (section
    3) cannot. `baseline_idx` is a column-index array, e.g. from select_features."""
    X, y, t, w, g, feature_names = cache
    f2f = feature_to_family()
    name_to_idx = {nm: i for i, nm in enumerate(feature_names)}
    baseline_idx = np.asarray(baseline_idx)
    baseline_names = {feature_names[i] for i in baseline_idx}
    baseline_families = {f2f[nm] for nm in baseline_names}

    if candidates == "families":
        candidate_families = sorted({f.name for f in REGISTRY} - {"time"})   # t_step/t_log always kept
    else:
        candidate_families = list(candidates)

    base_fold = cv_evaluate(X=X, y=y, t=t, g=g, feature_idx=baseline_idx, feature_names=feature_names,
                             n_folds=n_folds, seed=seed, num_rounds=num_rounds, max_series=max_series,
                             verbose=False)
    base_mean, base_std = float(base_fold.mean()), float(base_fold.std())
    if verbose:
        print("baseline CV TS-AUC: %.5f +/- %.5f (%d features)\n" % (base_mean, base_std, len(baseline_idx)))

    rows = [("__baseline__", base_mean, base_std, 0.0, len(baseline_idx))]
    for fam in candidate_families:
        in_baseline = fam in baseline_families
        if in_baseline:
            mod_names = {nm for nm in baseline_names if f2f[nm] != fam}
            action = "drop"
        else:
            fam_names = {nm for nm in feature_names if f2f[nm] == fam}
            mod_names = baseline_names | fam_names
            action = "add"
        if not mod_names:
            continue
        mod_idx = np.array(sorted(name_to_idx[nm] for nm in mod_names), dtype=np.int64)
        fold_aucs = cv_evaluate(X=X, y=y, t=t, g=g, feature_idx=mod_idx, feature_names=feature_names,
                                 n_folds=n_folds, seed=seed, num_rounds=num_rounds, max_series=max_series,
                                 verbose=False)
        mean_, std_ = float(fold_aucs.mean()), float(fold_aucs.std())
        delta = mean_ - base_mean
        label = "%s %s" % (action, fam)
        rows.append((label, mean_, std_, delta, len(mod_idx)))
        if verbose:
            print("%-16s | CV %.5f +/- %.5f | delta %+.5f | n_features %d" % (label, mean_, std_, delta, len(mod_idx)))

    return pd.DataFrame(rows, columns=["change", "cv_mean", "cv_std", "delta_mean", "n_features"])


## 3d. Research loop -- one call per idea

`research_step` is the whole keep/drop decision for one candidate family, in one call:
smoke test -> extract only what is missing -> univariate TS-AUC vs. the permutation-null
cutoff -> paired K-fold CV against the current registry. Minutes, not hours, because
every expensive thing it touches is cached by section 2b.

```python
class MyFamily(TwoSampleFamily): ...
research_step(MyFamily())                  # minutes, not hours
research_step(MyFamily(), register=True)   # keep it
local_test()                               # occasional end-to-end check
```

**What each stage costs on a repeat call.** Extraction: only the candidate family
(everything else is a `.npy` load). Univariate: the permutation null is computed once
per (sample, baseline, n_perm, seed) and cached in `null.joblib`, so the cutoff is a
dict lookup. CV: the baseline's per-fold AUCs are cached by (feature set, folds, LGB
params), so only the candidate model is trained.

**The folds are fixed by `seed`, not resampled.** Baseline and candidate see exactly
the same series in exactly the same folds, so the per-fold differences are *paired* and
the right noise estimate is the std of the per-fold deltas -- not the std of either
arm's AUCs, which is dominated by fold-to-fold difficulty that cancels in the pairing.
The verdict line uses that paired std.

`fast_lgb=True` (the default) uses `FAST_LGB` -- `learning_rate=0.1`, `max_bin=63`,
`num_boost_round=400`, `early_stopping(30)` -- which is for *ranking candidates against
each other*, not for estimating the final score. Re-run the finalists with
`fast_lgb=False` (`FINAL_LGB`: the real `LGB_PARAMS` / `NUM_ROUNDS`) before committing.

A `KEEP` verdict here is evidence, not proof: it is one seed, one sample of
`n_series`, and a subsampled validation set (section 3b's note applies). Confirm a real
decision with `fast_lgb=False`, a second `seed`, and finally `local_test()`.


In [25]:
# =====================================================================
#  3d. RESEARCH LOOP
# =====================================================================
import time

# Two named fit configurations. `overrides` are applied on top of LGB_PARAMS, which is
# defined in section 4 -- read at call time, never at import time, so this cell can sit
# above it.
# @crunch/keep:on
FAST_LGB = dict(
    overrides=dict(learning_rate=0.1, max_bin=63),
    num_boost_round=400,
    early_stopping=30,
)
# @crunch/keep:off
# @crunch/keep:on
FINAL_LGB = dict(
    overrides={},
    num_boost_round=None,        # None -> NUM_ROUNDS
    early_stopping=80,
)
# @crunch/keep:off


def _lgb_config(fast_lgb=True):
    cfg = FAST_LGB if fast_lgb else FINAL_LGB
    params = dict(LGB_PARAMS)
    params.update(cfg["overrides"])
    return params, cfg["num_boost_round"] or NUM_ROUNDS, cfg["early_stopping"]


def _sample_for(datasets, n_series, seed):
    """The same seeded sample run_feature_diagnostic draws, so a research step scores
    its candidate on the series the diagnostic reported on."""
    items = list(datasets)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(items), size=min(n_series, len(items)), replace=False)
    return [items[i] for i in idx]


def smoke_test_family(fam, datasets, n_series=3, verbose=True):
    """Cheap sanity checks before a candidate costs a full extraction.

    Catches the four ways a new family wastes an hour: duplicate names inside itself,
    a name that collides with an existing feature, a name/width disagreement, and a
    column that is constant (which scores exactly 0.500000 and looks like 'no signal')."""
    names = list(fam.names())
    if len(names) != len(set(names)):
        dupes = sorted({n for n in names if names.count(n) > 1})
        raise AssertionError("%s.names() has duplicates: %s" % (fam.name, dupes))

    existing_families = {f.name for f in REGISTRY}
    if fam.name in existing_families:
        raise AssertionError(
            "family name %r is already in REGISTRY -- pick another .name, or drop the "
            "old one first with research_step(..., drop={%r})" % (fam.name, fam.name))
    clash = sorted(set(names) & set(feature_to_family()))
    if clash:
        raise AssertionError(
            "%s emits feature name(s) that already exist: %s" % (fam.name, clash))

    reg = list(REGISTRY) + [fam]
    rows = []
    for did, xh, xo, tau in list(datasets)[:n_series]:
        fx = StreamFeatures(xh, active_families={fam.name}, registry=reg)
        if fx.names != names:
            raise AssertionError(
                "%s: driver emits %d columns, names() promises %d" % (fam.name, len(fx.names), len(names)))
        rows.append(np.asarray([fx.update(v) for v in xo], dtype=np.float64))
    R = np.concatenate(rows)

    if R.shape[1] != len(names):
        raise AssertionError("%s.update() returned %d values, names() has %d" % (fam.name, R.shape[1], len(names)))
    bad = ~np.isfinite(R)
    if bad.any():
        cols = [names[c] for c in np.unique(np.nonzero(bad)[1])]
        raise AssertionError("%s produced non-finite values in: %s" % (fam.name, cols))
    const = [names[c] for c in range(R.shape[1]) if R[:, c].std() == 0]
    if const:
        raise AssertionError(
            "%s produced CONSTANT column(s) over %d smoke rows: %s -- a constant column "
            "scores exactly 0.5 and is indistinguishable from 'no signal' in the "
            "diagnostic, so fix it before spending an extraction on it" % (fam.name, R.shape[0], const))

    if verbose:
        print("  smoke test OK: %d column(s), %d rows over %d series, all finite, none constant" % (R.shape[1], R.shape[0], min(n_series, len(list(datasets)))))
    return R


def _fixed_folds(g, n_folds, seed):
    """Series-level folds that depend only on (series set, n_folds, seed) -- so the
    baseline and the candidate are always scored on identical splits."""
    uniq = np.unique(g)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(uniq))
    return [tuple(sorted(uniq[ix].tolist())) for ix in np.array_split(perm, n_folds)]


def _cv_fold_aucs(X, y, t, w, g, names, idx, folds, params, num_rounds, early_stop):
    Xsel = X[:, idx]
    sel = [names[i] for i in idx]
    out = []
    for va_ids in folds:
        va = np.isin(g, np.asarray(va_ids))
        tr = ~va
        dtr = lgb.Dataset(Xsel[tr], label=y[tr], weight=w[tr],
                          feature_name=sel, free_raw_data=False)
        dva = lgb.Dataset(Xsel[va], label=y[va], feature_name=sel,
                          reference=dtr, free_raw_data=False)
        booster = lgb.train(
            params, dtr, num_boost_round=num_rounds,
            valid_sets=[dva], valid_names=["valid"],
            callbacks=[lgb.early_stopping(early_stop, verbose=False), lgb.log_evaluation(0)],
        )
        pva = booster.predict(Xsel[va], num_iteration=booster.best_iteration)
        out.append(ts_auc(y[va], t[va], pva))
    return np.asarray(out, dtype=np.float64)


def _cached_null(skey, X, y, t, g, names, fam_names, n_perm, seed, ranker, cache_dir,
                  verbose=True):
    """Permutation-null cutoff over one representative feature per BASELINE family.

    Keyed on the baseline family set (not the candidate's), so adding a family does not
    invalidate the cutoff and every later research_step is a dict lookup."""
    key = "null::" + _short_hash({"families": sorted(fam_names), "n_perm": n_perm,
                                   "seed": seed, "n_rows": int(len(y))})
    hit = load_null(skey, key, cache_dir=cache_dir)
    if hit is not None:
        if verbose:
            print("  null: cached (cutoff %.5f, %d perms)" % (hit["cutoff"], hit["n_perm"]))
        return hit["cutoff"], hit["null_df"]

    f2f = feature_to_family()
    base_cols = [i for i, nm in enumerate(names) if f2f.get(nm) in fam_names]
    tab = _folded_ts_auc_table(X[:, base_cols], y, t, [names[i] for i in base_cols],
                                ranker=ranker)
    tab["family"] = tab["feature"].map(f2f)
    rep = tab.sort_values("ts_auc_abs", ascending=False).drop_duplicates(subset="family")
    perm_features = rep["feature"].tolist()
    perm_idx = [names.index(nm) for nm in perm_features]
    if verbose:
        print("  null: computing over %d representative features x %d perms ..." % (len(perm_features), n_perm))
    null_df = permutation_null_ts_auc(X, y, t, g, perm_idx, perm_features,
                                       n_perm=n_perm, seed=seed, ranker=ranker)
    cutoff = float(null_df["null_p95"].max())
    save_null(skey, key, {"cutoff": cutoff, "null_df": null_df, "n_perm": n_perm},
              cache_dir=cache_dir)
    return cutoff, null_df


def research_step(new_family=None, drop=None, n_series=None, seed=0,
                   n_folds=3, fast_lgb=True, register=False,
                   datasets=None, n_perm=100, cache_dir=CACHE_DIR, verbose=True):
    """Score one idea against the current registry and print a verdict.

    new_family : a FeatureFamily INSTANCE to evaluate, or None to only measure `drop`.
    drop       : family names to remove from the baseline.
    register   : keep `new_family` in REGISTRY afterwards (default: restore the registry).

    Returns the univariate table for the candidate's columns; the CV numbers, the
    verdict and the stage timings are also on `df.attrs`.
    """
    datasets = train_data if datasets is None else datasets
    n_series = DIAG_N_SERIES if n_series is None else n_series
    drop = set(drop or ())
    sample = _sample_for(datasets, n_series, seed)

    registry_backup = list(REGISTRY)
    timings = {}
    try:
        # ---------------------------------------------------- 0. smoke test
        if new_family is not None:
            if verbose:
                print("== smoke test: %s (%d columns) ==" % (new_family.name,
                                                             len(new_family.names())))
            t0 = time.perf_counter()
            smoke_test_family(new_family, sample, n_series=3, verbose=verbose)
            timings["smoke"] = time.perf_counter() - t0
            REGISTRY.append(new_family)

        base_families = {f.name for f in REGISTRY} - drop
        if new_family is not None:
            base_families -= {new_family.name}
            cand_families = base_families | {new_family.name}
            label = "+%s" % new_family.name
        else:
            cand_families = base_families | drop
            label = "+%s" % ",".join(sorted(drop)) if drop else "(no change)"
        assert cand_families != base_families or not drop, \
            "nothing to compare: pass new_family= and/or drop="

        # ---------------------------------------------------- 1. extraction
        if verbose:
            print("\n== extraction (%d series, seed %d) ==" % (len(sample), seed))
        t0 = time.perf_counter()
        X, y, t, w, g, names = get_cached_matrix(sample, subsample=True, seed=seed,
                                                  cache_dir=cache_dir, verbose=verbose)
        timings["extract"] = time.perf_counter() - t0
        skey = sample_key([r[0] for r in sample], True, seed)
        f2f = feature_to_family()

        # ---------------------------------------------------- 2. univariate
        if verbose:
            print("\n== univariate ==")
        t0 = time.perf_counter()
        ranker = TSRanker(t)
        cutoff, _null_df = _cached_null(skey, X, y, t, g, names, base_families,
                                         n_perm, seed, ranker, cache_dir, verbose)
        if new_family is not None:
            cols = [i for i, nm in enumerate(names) if f2f.get(nm) == new_family.name]
            uni = _folded_ts_auc_table(X[:, cols], y, t, [names[i] for i in cols],
                                        ranker=ranker)
        else:
            uni = _folded_ts_auc_table(X[:, :0], y, t, [], ranker=ranker)
        uni["cutoff"] = cutoff
        uni["above_null"] = uni["ts_auc_abs"] >= cutoff
        uni = uni.sort_values("ts_auc_abs", ascending=False).reset_index(drop=True)
        timings["univariate"] = time.perf_counter() - t0
        if verbose and len(uni):
            print(uni.to_string(index=False))
            print("  %d/%d of the candidate's columns clear the null cutoff %.5f" % (int(uni["above_null"].sum()), len(uni), cutoff))

        # ---------------------------------------------------- 3. CV
        if verbose:
            print("\n== %d-fold CV (folds fixed by seed=%d, %s) ==" % (n_folds, seed, "FAST_LGB" if fast_lgb else "FINAL_LGB"))
        t0 = time.perf_counter()
        params, n_rounds, early = _lgb_config(fast_lgb)
        folds = _fixed_folds(g, n_folds, seed)

        base_idx = np.array([i for i, nm in enumerate(names)
                             if f2f.get(nm) in base_families], dtype=np.int64)
        cand_idx = np.array([i for i, nm in enumerate(names)
                             if f2f.get(nm) in cand_families], dtype=np.int64)

        fit_key = {"params": {k: params[k] for k in sorted(params)},
                   "num_rounds": n_rounds, "early": early,
                   "folds": [list(f) for f in folds]}
        base_key = "cv::" + _short_hash(dict(fit_key, features=[names[i] for i in base_idx]))
        base_fold = load_null(skey, base_key, cache_dir=cache_dir)
        if base_fold is None:
            base_fold = _cv_fold_aucs(X, y, t, w, g, names, base_idx, folds,
                                       params, n_rounds, early)
            save_null(skey, base_key, base_fold, cache_dir=cache_dir)
            base_src = "trained"
        else:
            base_src = "cached"
        cand_key = "cv::" + _short_hash(dict(fit_key, features=[names[i] for i in cand_idx]))
        cand_fold = load_null(skey, cand_key, cache_dir=cache_dir)
        if cand_fold is None:
            cand_fold = _cv_fold_aucs(X, y, t, w, g, names, cand_idx, folds,
                                       params, n_rounds, early)
            save_null(skey, cand_key, cand_fold, cache_dir=cache_dir)
            cand_src = "trained"
        else:
            cand_src = "cached"
        timings["cv"] = time.perf_counter() - t0

        deltas = cand_fold - base_fold
        mean_delta = float(deltas.mean())
        paired_std = float(deltas.std(ddof=1)) if len(deltas) > 1 else 0.0
        keep = mean_delta > paired_std

        if verbose:
            print("  baseline  (%2d families, %3d features, %s): %s | mean %.5f" % (len(base_families), len(base_idx), base_src,
                     " ".join("%.5f" % a for a in base_fold), base_fold.mean()))
            print("  candidate (%2d families, %3d features, %s): %s | mean %.5f" % (len(cand_families), len(cand_idx), cand_src,
                     " ".join("%.5f" % a for a in cand_fold), cand_fold.mean()))
            print("  per-fold delta: %s" % " ".join("%+.5f" % d for d in deltas))
            print("  mean delta %+.5f | paired std %.5f (ddof=1)" % (mean_delta, paired_std))
            print()
            print("  %-9s %s  mean delta %+.5f vs paired std %.5f   "
                  "[smoke %.1fs | extract %.1fs | univariate %.1fs | cv %.1fs]" % ("KEEP" if keep else "NO GAIN", label, mean_delta, paired_std,
                     timings.get("smoke", 0.0), timings["extract"],
                     timings["univariate"], timings["cv"]))
            if register and new_family is not None:
                print("  registered %r -- REGISTRY is now %d families" % (new_family.name, len(REGISTRY)))
            elif new_family is not None:
                print("  registry restored (pass register=True to keep %r)" % new_family.name)

        uni.attrs.update(
            verdict="KEEP" if keep else "NO GAIN",
            label=label,
            baseline_folds=base_fold, candidate_folds=cand_fold, deltas=deltas,
            mean_delta=mean_delta, paired_std=paired_std, cutoff=cutoff,
            timings=timings, folds=folds, sample_key=skey,
            baseline_families=sorted(base_families),
            candidate_families=sorted(cand_families),
        )
        return uni
    finally:
        if not (register and new_family is not None):
            REGISTRY[:] = registry_backup


def local_test(**kwargs):
    """The real end-to-end crunch run: train() + infer() + determinism check on the
    actual competition data. Minutes to hours -- this is the occasional final smoke
    test, not part of the iteration loop. Everything above exists so that this is
    rarely the thing you are waiting on."""
    return crunch_tools.test(**kwargs)


print("research_step / local_test defined -- see the markdown above for the 4-line workflow.")


research_step / local_test defined -- see the markdown above for the 4-line workflow.


## 4. `train()`

Binary logloss, group-split by series id. Early stopping on validation logloss, then the real
TS-AUC is reported on the held-out series at full step resolution.

Knobs worth sweeping first: `num_leaves`, `min_data_in_leaf`, and the training-row weight
scheme (`USE_METRIC_WEIGHTS`).

**Train/serve parity, section 5's real risk.** `train()` here is the crunch-submission path, so
it extracts through `build_dataset`'s default (`active_families=_USE_GLOBAL_ACTIVE`, i.e.
whatever `ACTIVE_FAMILIES` currently is) -- the same subset `infer()` will compute, not the full
registry. This is deliberate: `ACTIVE_FAMILIES` is decided offline via the section 3c iteration
loop (which always works off the full-registry cache), and once decided, `train()`/`infer()`
both only ever pay for that subset -- keeping them fast and, more importantly, keeping them
*the same feature set*, which is the actual train/serve-skew risk this refactor introduces (a
model trained on one family list must never be scored with a `StreamFeatures` built for a
different one). The artifact now stores the resolved active-family list and the exact feature
list, and `infer()` reconstructs its extractor from *that*, not from whatever the notebook's
global happens to be at inference time -- see the parity-check cell at the end of section 5.


In [2]:
# @crunch/keep:on
LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.04,
    num_leaves=63,
    min_data_in_leaf=300,
    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,
    lambda_l2=5.0,
    max_bin=127,
    num_threads=max(1, (os.cpu_count() or 4) - 1),
    verbosity=-1,
)
LGB_PARAMS.update(
    learning_rate=0.043114,
    num_leaves=45,
    min_data_in_leaf=794,
    feature_fraction=0.539909,
    bagging_fraction=0.805950,
    lambda_l2=79.616230,
    extra_trees=True,
    min_gain_to_split=0.408390,
)
NUM_ROUNDS = 3000
VALID_FRACTION = 0.15
TRAIN_MAX_SERIES = None      # set an int for fast iteration
SEEDS = (0, 1, 2)
CAT_PARAMS = dict(
    grow_policy="Depthwise",
    learning_rate=0.028354684013833883,
    depth=8,
    l2_leaf_reg=12.76103092455787,
    rsm=0.5758405578310992,
    random_strength=2.5835125442520925,
    subsample=0.7933098848053978,
    min_data_in_leaf=64,
    bootstrap_type="Bernoulli",
)
CAT_WEIGHT = 0.3          # p = (1 - CAT_WEIGHT) * mean(lgb seeds) + CAT_WEIGHT * catboost
# @crunch/keep:off


def active_feature_names(active_families=None) -> List[str]:
    """The feature name list StreamFeatures(..., active_families=active_families)
    actually emits, in order -- FEATURES itself only covers active_families=None
    (the full registry). train()/infer() must always pass their booster's
    feature_name= this, never the bare FEATURES global, whenever ACTIVE_FAMILIES
    is a strict subset."""
    return [nm for f in REGISTRY if active_families is None or f.name in active_families
            for nm in f.names()]


def train(datasets, model_directory_path: str):
    items = list(datasets)
    if TRAIN_MAX_SERIES:
        items = items[:TRAIN_MAX_SERIES]

    ids = np.array([r[0] for r in items])
    rng = np.random.default_rng(0)
    is_val = rng.random(len(items)) < VALID_FRACTION

    tr_items = [r for r, v in zip(items, is_val) if not v]
    va_items = [r for r, v in zip(items, is_val) if v]
    print("train series %d | valid series %d" % (len(tr_items), len(va_items)))

    # active_families left at the sentinel default -> resolves to the current
    # ACTIVE_FAMILIES global, so this is exactly what infer() will compute too.
    Xtr, ytr, ttr, wtr, _ = build_dataset(tr_items, subsample=True)
    Xva, yva, tva, _, gva = build_dataset(va_items, subsample=False)
    print("rows: train %s  valid %s" % (Xtr.shape, Xva.shape))

    feat_names = active_feature_names(ACTIVE_FAMILIES)
    assert Xtr.shape[1] == len(feat_names), (
        "extracted %d columns but active_feature_names() expects %d -- "
        "ACTIVE_FAMILIES changed between extraction and here?" % (Xtr.shape[1], len(feat_names)))

    dtr = lgb.Dataset(Xtr, label=ytr, weight=wtr, feature_name=feat_names, free_raw_data=False)
    dva = lgb.Dataset(Xva, label=yva, feature_name=feat_names, reference=dtr, free_raw_data=False)

    boosters, iters = [], []
    for s in SEEDS:
        params = dict(LGB_PARAMS)
        params.update(seed=s, bagging_seed=s + 100,
                      feature_fraction_seed=s + 200)
        b = lgb.train(params, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(100)])
        boosters.append(b); iters.append(b.best_iteration)
        print("seed %d: %d trees" % (s, b.best_iteration))

    # CatBoost on exactly the same rows, labels and stride weights
    cat = CatBoostClassifier(**CAT_PARAMS, iterations=NUM_ROUNDS, early_stopping_rounds=100,
                             random_seed=0, thread_count=8, verbose=0, allow_writing_files=False)
    cat.fit(Xtr, ytr, sample_weight=wtr, eval_set=(Xva, yva))
    cat_iters = int(cat.get_best_iteration())
    print("catboost: best iteration %d" % cat_iters)

    p_lgb = np.mean([b.predict(Xva, num_iteration=it) for b, it in zip(boosters, iters)], axis=0)
    p_cat = cat.predict_proba(Xva)[:, 1]
    pva = (1.0 - CAT_WEIGHT) * p_lgb + CAT_WEIGHT * p_cat
    print("HOLD-OUT TS-AUC lightgbm : %.5f" % ts_auc(yva, tva, p_lgb))
    print("HOLD-OUT TS-AUC catboost : %.5f" % ts_auc(yva, tva, p_cat))
    print("HOLD-OUT TS-AUC: %.5f  (blend, cat weight %.2f)" % (ts_auc(yva, tva, pva), CAT_WEIGHT))

    np.savez_compressed(os.path.join(model_directory_path, "holdout_scores.npz"),
                        p=pva, y=yva, t=tva, g=gva)
    print("saved hold-out scores -> model dir")

    imp = pd.Series(boosters[0].feature_importance("gain"), index=feat_names).sort_values(ascending=False)
    print(imp.head(25))

    # Model artifact is self-describing: it carries the exact active-family set
    # and feature list it was trained with, so infer() can reconstruct the same
    # extractor from the artifact itself rather than trusting the notebook's
    # ACTIVE_FAMILIES global at inference time (which could have drifted).
    joblib.dump(
        {
            "boosters": boosters,
            "features": feat_names,
            "active_families": sorted(ACTIVE_FAMILIES) if ACTIVE_FAMILIES else None,
            "best_iterations": iters,
            "catboost": cat,
            "cat_weight": CAT_WEIGHT,
            "cat_iters": cat_iters,
        },
        os.path.join(model_directory_path, "model.joblib"),
    )


NameError: name 'os' is not defined

## 5. `infer()`

**Watch the time budget.** Up to 10M scores means the per-step cost of `booster.predict` on a
single row (~50-100 us) dominates everything. Two safe levers:

- `PREDICT_STRIDE`: recompute the LightGBM prediction only every `K` steps, where `K` depends
  **only on `t`**. Because every series shares the schedule, all series at step `t` are equally
  stale, so within-`t` ranking is unaffected — this is free speed at large `t`.
- Keep `best_iteration` modest (300-600 trees) and `num_threads=1` inside `infer` when running
  with `INFER_PARALLELISM > 1`.

`SMOOTH_ALPHA` / `RUNNING_MAX` are post-processing switches worth ablating: the label is
monotone non-decreasing in `t` for a given series, so removing downward noise sometimes helps.

**`infer()` reconstructs its extractor from the artifact, not from the notebook's live
`ACTIVE_FAMILIES`.** It loads `art["active_families"]` and builds `StreamFeatures(x_historical,
active_families=art["active_families"])` -- so a stale/edited global in this notebook can never
silently desync what a *loaded* model actually predicts on. This also means `infer()` only ever
computes the family subset the model was trained with (never the full registry), which is what
keeps it inside the prediction time budget regardless of how large the registry grows.


In [27]:
# @crunch/keep:on
SMOOTH_ALPHA = 0.0     # 0 = off; e.g. 0.3 = EWMA-smooth the score across t
RUNNING_MAX = False    # enforce monotone non-decreasing score
# @crunch/keep:off


def _stride_for(t: int) -> int:
    if t < 64:
        return 1
    if t < 256:
        return 2
    return 4


def infer(datasets, model_directory_path: str):
    art = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    boosters = art.get("boosters") or [art["booster"]]
    best_its = art.get("best_iterations") or [art["best_iteration"]]
    for b in boosters:
        b.params["num_threads"] = 1
    active_families = set(art["active_families"]) if art.get("active_families") else None
    cat = art.get("catboost"); cat_w = float(art.get("cat_weight", 0.0))
    n_feat = len(art["features"])

    buf = np.empty((1, n_feat), dtype=np.float64)

    yield  # signal readiness

    for x_historical, x_online in datasets:
        fx = StreamFeatures(x_historical, active_families=active_families)
        prev = 0.0
        smoothed = 0.0
        running = 0.0

        for t, point in enumerate(x_online):
            f = fx.update(point)

            if t % _stride_for(t) == 0:
                buf[0, :] = f
                pl = float(np.mean([b.predict(buf, num_iteration=it, num_threads=1)[0]
                                    for b, it in zip(boosters, best_its)]))
                prev = pl if cat is None else (1.0 - cat_w) * pl + cat_w * float(cat.predict_proba(buf.copy(), thread_count=1)[0, 1])
                # .copy(): catboost marks any array it predicts on as read-only, and buf is reused
            p = prev

            if SMOOTH_ALPHA > 0.0:
                smoothed = (1.0 - SMOOTH_ALPHA) * smoothed + SMOOTH_ALPHA * p if t else p
                p = smoothed
            if RUNNING_MAX:
                running = p if p > running else running
                p = running

            if p < 0.0:
                p = 0.0
            elif p > 1.0:
                p = 1.0
            yield float(p)


### Train/infer parity check -- run this after training a model, before trusting `infer()`

This is the actual train/serve-skew risk the registry refactor introduces (the old code used
one hardcoded `StreamFeatures`, so this couldn't drift; now the extractor is parameterized by
`active_families`, so it's an explicit thing to verify rather than assume). Loads the just-
trained artifact, builds a `StreamFeatures` the way `infer()` does (from `art["active_families"]`)
and the way `train()`'s extraction path does (via `active_feature_names(ACTIVE_FAMILIES)` +
`build_dataset`'s default), and asserts they produce identical feature vectors on the same
handful of series.


In [28]:
def check_train_infer_parity(model_directory_path, datasets, n_series=20):
    art = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    infer_active = set(art["active_families"]) if art.get("active_families") else None
    train_active = ACTIVE_FAMILIES
    assert infer_active == train_active, (
        "artifact's active_families (%s) != current ACTIVE_FAMILIES (%s) -- "
        "you're checking parity against the wrong global; re-set ACTIVE_FAMILIES "
        "to what this artifact was trained with." % (infer_active, train_active))
    assert art["features"] == active_feature_names(infer_active), (
        "artifact's stored feature list doesn't match active_feature_names(infer_active) -- "
        "REGISTRY must have changed since this model was trained.")

    items = list(datasets)[:n_series]
    max_abs_diff = 0.0
    for did, xh, xo, tau in items:
        fx_infer = StreamFeatures(xh, active_families=infer_active)     # infer()'s construction
        fx_train = StreamFeatures(xh, active_families=train_active)     # train()'s construction
        for point in xo:
            r_i = np.asarray(fx_infer.update(point), dtype=np.float64)
            r_t = np.asarray(fx_train.update(point), dtype=np.float64)
            max_abs_diff = max(max_abs_diff, float(np.max(np.abs(r_i - r_t))))
            assert np.allclose(r_i, r_t, atol=1e-9, rtol=1e-9), (
                "train/infer feature mismatch for series %s" % did)
    print("PARITY OK over %d series, max abs diff = %.3e" % (len(items), max_abs_diff))


# Run after train() (section 4) / crunch_tools.test() (section 6) has produced model.joblib.
# Fill in the actual model_directory_path crunch used (crunch_tools.test() prints/creates it,
# commonly "resources" or wherever `train()`'s model_directory_path argument pointed):
# check_train_infer_parity("resources", train_data, n_series=20)


In [53]:
import os
os.makedirs("resources", exist_ok=True)
train(train_data, "resources")

train series 8469 | valid series 1531


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done 260 tasks      | elapsed:    4.7s
[Parallel(n_jobs=-1)]: Done 1460 tasks      | elapsed:   22.1s
[Parallel(n_jobs=-1)]: Done 3460 tasks      | elapsed:   53.9s
[Parallel(n_jobs=-1)]: Done 6260 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 8469 out of 8469 | elapsed:  2.4min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done 260 tasks      | elapsed:    6.0s
[Parallel(n_jobs=-1)]: Done 1425 tasks      | elapsed:   30.8s
[Parallel(n_jobs=-1)]: Done 1531 out of 1531 | elapsed:   32.8s finished


rows: train (1302145, 85)  valid (757457, 85)
[100]	valid_0's binary_logloss: 0.507248
[200]	valid_0's binary_logloss: 0.506145
seed 0: 189 trees
[100]	valid_0's binary_logloss: 0.507519
[200]	valid_0's binary_logloss: 0.506444
seed 1: 187 trees
[100]	valid_0's binary_logloss: 0.507197
[200]	valid_0's binary_logloss: 0.506608
seed 2: 159 trees
HOLD-OUT TS-AUC: 0.60631
saved hold-out scores -> model dir
t_log                1.176967e+06
t_step               9.368519e+05
bocpd_logodds        5.945596e+05
bocpd_runlen_log     4.890576e+05
e2_exp               2.168640e+05
max_absz_exp         1.831061e+05
tail3_w64            1.593692e+05
ftest_e_w256         1.586505e+05
beyond_range_exp     1.463834e+05
lmom_lkurt_z_w64     1.216682e+05
e2_w256              1.134614e+05
lmom_lkurt_z_w256    1.021287e+05
tail3_exp            9.690762e+04
mood_e_w256          9.608749e+04
lmom_lkurt_e_w256    9.361068e+04
e2_ew0.02            8.648279e+04
tail2_exp            8.364679e+04
bocpd_logvar    

In [29]:
# @crunch/keep:on
INFER_PARALLELISM = 4
# @crunch/keep:off

## 6. Local test + leaderboard metric


In [ ]:
crunch_tools.test(
    # force_first_train=False,
    # no_determinism_check=True,
)


In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
y_test = pd.read_parquet("data/y_test.reduced.parquet")

merged = prediction.merge(y_test, how="left", left_index=True, right_index=True)
merged["time_online"] = merged.groupby("id").cumcount()

print("Local TS-AUC: %.5f" % ts_auc(
    merged["target"].values, merged["time_online"].values, merged["prediction"].values
))


In [35]:
def train_full(datasets, model_directory_path, round_scale=1.18, rounds=None, seeds=None):
    """Refit on ALL series with a fixed tree count taken from the early-stopped run.
    No validation split, so nothing is held back from fitting."""
    path = os.path.join(model_directory_path, "model.joblib")
    prev = joblib.load(path)                       # the early-stopped artifact written by train()
    if rounds is None:
        base = int(np.mean(prev["best_iterations"]))
        rounds = int(round(base * round_scale))
        print("early-stopped trees %d -> fixed rounds %d (scale %.2f)" % (base, rounds, round_scale))
    shutil.copy(path, path + ".holdout_backup")

    items = list(datasets)
    X, y, t, w, g = build_dataset(items, subsample=True)
    feat_names = active_feature_names(ACTIVE_FAMILIES)
    print("full-fit rows:", X.shape, "| series:", len(items))

    dtr = lgb.Dataset(X, label=y, weight=w, feature_name=feat_names)
    boosters, iters = [], []
    for s in (seeds or SEEDS):
        params = dict(LGB_PARAMS)
        params.update(seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
        b = lgb.train(params, dtr, num_boost_round=rounds)
        boosters.append(b); iters.append(rounds)
        print("seed %d done" % s)

    cat, cat_rounds = None, None
    if prev.get("cat_iters"):
        cat_rounds = int(round(prev["cat_iters"] * round_scale))
        cat = CatBoostClassifier(**CAT_PARAMS, iterations=cat_rounds, random_seed=0,
                                 thread_count=8, verbose=0, allow_writing_files=False)
        cat.fit(X, y, sample_weight=w)
        print("catboost: %d early-stopped -> %d iterations (scale %.2f)" % (prev["cat_iters"], cat_rounds, round_scale))
    else:
        print("previous artifact has no cat_iters -- refitting LightGBM only")

    joblib.dump(
        {
            "boosters": boosters,
            "features": feat_names,
            "active_families": sorted(ACTIVE_FAMILIES) if ACTIVE_FAMILIES else None,
            "best_iterations": iters,
            "catboost": cat,
            "cat_weight": CAT_WEIGHT if cat is not None else 0.0,
            "cat_iters": cat_rounds,
        },
        path,
    )
    print("wrote", path, "| trees per booster:", rounds)

In [41]:
train_full(train_data, "resources", rounds=270)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done 260 tasks      | elapsed:    3.4s


KeyboardInterrupt: 

NameError: name 'FOLD_IDX' is not defined